# C4_markowitz_portfolio_optimization: Markowitz portfolio optimization

Public appendix notebook from the Bachelor's Thesis *Evaluación robusta de estrategias de inversión: backtesting, sobreajuste y validación estadística*.

This notebook is included for transparency/reproducibility. It was originally developed in Google Colab; paths may need to be adapted for local execution.


In [ ]:
# -*- coding: utf-8 -*-
"""block4.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1HYg173vJHmvfSI3dEZE6DoVW0xM8ssD-

Instalación
"""

!pip -q install pandas numpy yfinance matplotlib lxml html5lib beautifulsoup4 requests


In [ ]:
# ============================================================
# CELDA 1 - MONTAR GOOGLE DRIVE Y CREAR CARPETAS


In [ ]:
# ============================================================

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

# Carpeta base del TFG en Drive
TFG_BASE_DIR = "/content/drive/MyDrive/TFG"

# Crear carpetas necesarias
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

print("Google Drive montado.")
print(f"Carpeta base del TFG: {TFG_BASE_DIR}")
print("Carpetas creadas correctamente.")

"""Código completo del Bloque 0"""


In [ ]:
# ============================================================
# CAP 0 - MOTOR REPRODUCIBLE DE DATOS, BACKTESTING Y MÉTRICAS
# Versión Colab con reconstrucción histórica del S&P 100


In [ ]:
# ============================================================

from __future__ import annotations

import json
import math
import time
import warnings
import re
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from io import StringIO
from pathlib import Path
from typing import Iterable, List, Optional, Tuple

import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import requests


In [ ]:
# ============================================================
# 1. CONFIGURACIÓN GENERAL


In [ ]:
# ============================================================

@dataclass
class BacktestConfig:
    # Periodo de análisis.
    # Yahoo Finance usa end_date como fecha final EXCLUSIVA.
    start_date: str = "2021-01-01"
    end_date: str = "2026-01-01"

    # Datos
    data_source: str = "Yahoo Finance via yfinance"
    universe_name: str = "S&P 100"

    # Archivo manual si algún día tenemos una fuente histórica propia.
    membership_file: str = "data/sp100_membership.csv"

    # Archivo reconstruido automáticamente desde revisiones históricas de Wikipedia.
    reconstructed_membership_file: str = "data/sp100_membership_reconstructed_wikipedia.csv"
    snapshot_log_file: str = "data/sp100_wikipedia_snapshot_log.csv"

    # Reconstrucción histórica
    use_reconstructed_membership: bool = True
    force_rebuild_membership: bool = True
    membership_history_frequency: str = "MS"

    # Importante:
    # Si esto está en False, el código NO vuelve a una lista actual si falla la reconstrucción.
    # Así evitamos meter survivorship bias sin querer.
    allow_current_fallback: bool = False

    # Benchmark
    # SPY = proxy invertible del S&P 500 con precios ajustados.
    # ^GSPC = índice S&P 500 de precio, no total return.
    benchmark_ticker: str = "SPY"
    benchmark_name: str = "S&P 500 benchmark"

    # Backtest
    initial_capital: float = 1.0
    trading_days_per_year: int = 252

    # Costes
    transaction_cost: float = 0.001
    cost_sensitivity: Tuple[float, ...] = (0.0, 0.0005, 0.001, 0.0025)

    # Tasa libre de riesgo anual provisional.
    risk_free_rate_annual: float = 0.0225

    # Limpieza
    min_price_coverage: float = 0.80
    min_assets_required: int = 20

    # Estrategia
    # "static_buy_hold": compra al inicio el universo histórico inicial y no rebalancea.
    # "dynamic_equal_weight": usa el universo histórico correspondiente en cada rebalanceo.
    strategy_mode: str = "dynamic_equal_weight"
    rebalance_frequency: str = "M"

    # Reproducibilidad
    random_seed: int = 123
    output_dir: str = "results/block0_sp100_survivorship_corrected"


CONFIG = BacktestConfig()


In [ ]:
# ============================================================
# 2. UTILIDADES GENERALES


In [ ]:
# ============================================================

def ensure_dir(path: str | Path) -> Path:
    path = Path(path)
    path.mkdir(parents=True, exist_ok=True)
    return path


def now_utc_iso() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def save_json(obj: dict, path: str | Path) -> None:
    path = Path(path)
    ensure_dir(path.parent)
    with path.open("w", encoding="utf-8") as f:
        json.dump(obj, f, indent=4, ensure_ascii=False, default=str)


def to_yfinance_ticker(ticker: str) -> str:
    """
    Yahoo Finance usa '-' en vez de '.' para clases de acciones.
    Ejemplo: BRK.B -> BRK-B.
    """
    return str(ticker).strip().replace(".", "-")


def from_yfinance_ticker(ticker: str) -> str:
    return str(ticker).strip().replace("-", ".")


In [ ]:
# ============================================================
# 3. UNIVERSO S&P 100 HISTÓRICO


In [ ]:
# ============================================================

WIKIPEDIA_API_URL = "https://en.wikipedia.org/w/api.php"
WIKIPEDIA_TITLE_SP100 = "S&P 100"


# Lista solo de emergencia.
# Por defecto NO se usa porque allow_current_fallback=False.
SP100_FALLBACK_TICKERS = [
    "AAPL", "ABBV", "ABT", "ACN", "ADBE", "AIG", "AMD", "AMGN", "AMT", "AMZN",
    "AVGO", "AXP", "BA", "BAC", "BK", "BKNG", "BLK", "BMY", "BRK.B", "C",
    "CAT", "CHTR", "CL", "CMCSA", "COF", "COP", "COST", "CRM", "CSCO", "CVS",
    "CVX", "DE", "DHR", "DIS", "DUK", "EMR", "EXC", "F", "FDX", "GD",
    "GE", "GILD", "GM", "GOOG", "GOOGL", "GS", "HD", "HON", "IBM", "INTC",
    "JNJ", "JPM", "KHC", "KO", "LIN", "LLY", "LMT", "LOW", "MA", "MCD",
    "MDLZ", "MDT", "MET", "META", "MMM", "MO", "MRK", "MS", "MSFT", "NEE",
    "NFLX", "NKE", "NVDA", "ORCL", "PEP", "PFE", "PG", "PM", "PYPL", "QCOM",
    "RTX", "SBUX", "SCHW", "SO", "SPG", "T", "TGT", "TMO", "TXN", "UNH",
    "UNP", "UPS", "USB", "V", "VZ", "WBA", "WFC", "WMT", "XOM"
]


def clean_ticker_symbol(x: str) -> Optional[str]:
    """
    Limpia símbolos extraídos de tablas HTML.
    """
    if pd.isna(x):
        return None

    s = str(x).strip()
    s = re.sub(r"\[.*?\]", "", s)
    s = s.replace("\xa0", " ")
    s = s.strip()

    if not s:
        return None

    bad_values = {"symbol", "ticker", "nan", "none", "company", "security"}
    if s.lower() in bad_values:
        return None

    s = re.sub(r"[^A-Za-z0-9\.\-]", "", s)

    if not s:
        return None

    return s.upper()


def wikipedia_api_get(
    params: dict,
    max_retries: int = 4,
    sleep_seconds: float = 0.8,
) -> dict:
    """
    Petición robusta a la API de Wikipedia.
    """
    headers = {
        "User-Agent": "TFG-Backtest-Research/1.0 academic Colab notebook"
    }

    last_error = None

    for attempt in range(max_retries):
        try:
            response = requests.get(
                WIKIPEDIA_API_URL,
                params=params,
                headers=headers,
                timeout=30,
            )

            response.raise_for_status()
            data = response.json()

            if "error" in data:
                raise RuntimeError(data["error"])

            return data

        except Exception as e:
            last_error = e
            time.sleep(sleep_seconds * (attempt + 1))

    raise RuntimeError(f"No se pudo consultar Wikipedia API. Último error: {last_error}")


def get_wikipedia_revision_at(date: pd.Timestamp) -> Tuple[int, str]:
    """
    Obtiene la revisión de la página S&P 100 existente en una fecha concreta.
    """
    date = pd.Timestamp(date)
    rvstart = date.strftime("%Y-%m-%dT23:59:59Z")

    params = {
        "action": "query",
        "format": "json",
        "formatversion": "2",
        "prop": "revisions",
        "titles": WIKIPEDIA_TITLE_SP100,
        "rvprop": "ids|timestamp",
        "rvlimit": "1",
        "rvdir": "older",
        "rvstart": rvstart,
    }

    data = wikipedia_api_get(params)
    pages = data.get("query", {}).get("pages", [])

    if not pages or "revisions" not in pages[0] or not pages[0]["revisions"]:
        raise RuntimeError(f"No se encontró revisión de Wikipedia para {date.date()}")

    rev = pages[0]["revisions"][0]

    return int(rev["revid"]), str(rev["timestamp"])


def get_wikipedia_revision_html(revid: int) -> str:
    """
    Convierte una revisión concreta a HTML mediante action=parse.
    """
    params = {
        "action": "parse",
        "format": "json",
        "oldid": str(revid),
        "prop": "text",
    }

    data = wikipedia_api_get(params)
    html = data.get("parse", {}).get("text", {}).get("*")

    if not html:
        raise RuntimeError(f"No se pudo extraer HTML para oldid={revid}")

    return html


def extract_sp100_tickers_from_html(html: str) -> List[str]:
    """
    Extrae los tickers de la tabla de constituyentes del S&P 100.

    Busca tablas con columna Symbol/Ticker.
    Acepta una tabla si encuentra aproximadamente entre 80 y 120 símbolos.
    """
    tables = pd.read_html(StringIO(html))
    candidates = []

    for table in tables:
        table = table.copy()

        if isinstance(table.columns, pd.MultiIndex):
            table.columns = [
                " ".join([str(x) for x in col if str(x) != "nan"]).strip()
                for col in table.columns
            ]
        else:
            table.columns = [str(c).strip() for c in table.columns]

        symbol_col = None

        for col in table.columns:
            col_lower = str(col).lower()
            if "symbol" in col_lower or "ticker" in col_lower:
                symbol_col = col
                break

        if symbol_col is None:
            continue

        tickers = []

        for value in table[symbol_col].tolist():
            ticker = clean_ticker_symbol(value)
            if ticker is not None:
                tickers.append(ticker)

        tickers = sorted(set(tickers))

        if 80 <= len(tickers) <= 120:
            candidates.append(tickers)

    if not candidates:
        raise RuntimeError("No se encontró una tabla válida de constituyentes del S&P 100.")

    candidates = sorted(candidates, key=lambda x: abs(len(x) - 100))

    return candidates[0]


def fetch_sp100_snapshot_from_wikipedia(date: pd.Timestamp) -> Tuple[List[str], dict]:
    """
    Descarga la composición del S&P 100 según la página de Wikipedia
    tal como estaba en la fecha indicada.
    """
    revid, revision_timestamp = get_wikipedia_revision_at(date)
    html = get_wikipedia_revision_html(revid)
    tickers = extract_sp100_tickers_from_html(html)

    metadata = {
        "snapshot_date": str(pd.Timestamp(date).date()),
        "wiki_revision_id": revid,
        "wiki_revision_timestamp": revision_timestamp,
        "n_tickers": len(tickers),
        "status": "ok",
    }

    return tickers, metadata


def make_snapshot_dates(start_date: str, end_date: str, freq: str) -> List[pd.Timestamp]:
    """
    Genera fechas de snapshot entre start_date y end_date.

    end_date se considera exclusiva, igual que en Yahoo Finance.
    """
    start = pd.Timestamp(start_date).normalize()
    end_exclusive = pd.Timestamp(end_date).normalize()
    end_inclusive = end_exclusive - pd.Timedelta(days=1)

    regular_dates = pd.date_range(start=start, end=end_inclusive, freq=freq)

    dates = sorted(set([start] + list(regular_dates)))
    dates = [pd.Timestamp(d).normalize() for d in dates if start <= d <= end_inclusive]

    return dates


def snapshots_to_membership(snapshot_rows: List[dict]) -> pd.DataFrame:
    """
    Convierte snapshots mensuales en intervalos de pertenencia.
    """
    snapshots = pd.DataFrame(snapshot_rows)

    if snapshots.empty:
        raise RuntimeError("No hay snapshots para construir membership.")

    snapshots["snapshot_date"] = pd.to_datetime(snapshots["snapshot_date"])
    snapshots = snapshots.sort_values("snapshot_date")

    interval_rows = []
    unique_dates = sorted(snapshots["snapshot_date"].unique())

    for i, date in enumerate(unique_dates):
        current = snapshots[snapshots["snapshot_date"] == date]
        tickers = sorted(current["ticker"].dropna().unique().tolist())

        if i < len(unique_dates) - 1:
            interval_end = pd.Timestamp(unique_dates[i + 1]) - pd.Timedelta(days=1)
        else:
            interval_end = pd.NaT

        for ticker in tickers:
            interval_rows.append({
                "ticker": ticker,
                "yf_ticker": to_yfinance_ticker(ticker),
                "start": pd.Timestamp(date),
                "end": interval_end,
                "source": "wikipedia_historical_revision_snapshot",
            })

    membership = pd.DataFrame(interval_rows)

    if membership.empty:
        raise RuntimeError("Membership histórico vacío.")

    return membership


def reconstruct_sp100_membership_from_wikipedia(
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Reconstruye la composición histórica del S&P 100 usando revisiones históricas
    de Wikipedia.

    Esto evita usar una lista actual para todo el pasado.
    """
    snapshot_dates = make_snapshot_dates(
        config.start_date,
        config.end_date,
        config.membership_history_frequency,
    )

    print("Reconstruyendo S&P 100 histórico desde Wikipedia.")
    print(f"Snapshots: {len(snapshot_dates)} fechas ({config.membership_history_frequency})")

    snapshot_rows = []
    snapshot_log = []

    last_good_tickers = None

    for i, date in enumerate(snapshot_dates):
        print(f"  Snapshot {i + 1}/{len(snapshot_dates)}: {date.date()}")

        try:
            tickers, meta = fetch_sp100_snapshot_from_wikipedia(date)
            last_good_tickers = tickers

        except Exception as e:
            if last_good_tickers is None:
                raise RuntimeError(
                    f"Falló el primer snapshot ({date.date()}) y no hay composición anterior para arrastrar. "
                    f"Error: {e}"
                )

            tickers = last_good_tickers
            meta = {
                "snapshot_date": str(date.date()),
                "wiki_revision_id": None,
                "wiki_revision_timestamp": None,
                "n_tickers": len(tickers),
                "status": f"carried_forward_after_error: {type(e).__name__}: {e}",
            }

        for ticker in tickers:
            snapshot_rows.append({
                "snapshot_date": str(date.date()),
                "ticker": ticker,
            })

        snapshot_log.append(meta)

        time.sleep(0.25)

    membership = snapshots_to_membership(snapshot_rows)
    snapshot_log_df = pd.DataFrame(snapshot_log)

    ensure_dir("data")

    membership.to_csv(config.reconstructed_membership_file, index=False)
    snapshot_log_df.to_csv(config.snapshot_log_file, index=False)

    return membership, snapshot_log_df


def load_membership_schedule(config: BacktestConfig) -> Tuple[pd.DataFrame, str]:
    """
    Carga o reconstruye el membership histórico.

    Orden:
    1. Si existe data/sp100_membership.csv manual, lo usa.
    2. Si existe membership reconstruido y force_rebuild_membership=False, lo usa.
    3. Si no existe, reconstruye desde revisiones históricas de Wikipedia.
    4. Si todo falla y allow_current_fallback=False, lanza error.
    """
    manual_path = Path(config.membership_file)
    reconstructed_path = Path(config.reconstructed_membership_file)

    if manual_path.exists():
        membership = pd.read_csv(manual_path)

        required = {"ticker", "start", "end"}
        missing = required - set(membership.columns)

        if missing:
            raise ValueError(f"Faltan columnas en {manual_path}: {missing}")

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "manual_historical_membership_file"

    if (
        config.use_reconstructed_membership
        and reconstructed_path.exists()
        and not config.force_rebuild_membership
    ):
        membership = pd.read_csv(reconstructed_path)

        membership["ticker"] = membership["ticker"].astype(str).str.strip()
        membership["yf_ticker"] = membership["ticker"].map(to_yfinance_ticker)
        membership["start"] = pd.to_datetime(membership["start"], errors="coerce")
        membership["end"] = pd.to_datetime(membership["end"], errors="coerce")

        return membership, "cached_wikipedia_reconstructed_historical_membership"

    if config.use_reconstructed_membership:
        try:
            membership, snapshot_log_df = reconstruct_sp100_membership_from_wikipedia(config)
            return membership, "wikipedia_reconstructed_historical_membership"

        except Exception as e:
            if not config.allow_current_fallback:
                raise RuntimeError(
                    "No se pudo reconstruir la composición histórica del S&P 100 y "
                    "allow_current_fallback=False. No se usará lista actual para evitar survivorship bias. "
                    f"Error original: {e}"
                )

    if config.allow_current_fallback:
        tickers = sorted(SP100_FALLBACK_TICKERS)

        membership = pd.DataFrame({
            "ticker": tickers,
            "yf_ticker": [to_yfinance_ticker(t) for t in tickers],
            "start": pd.Timestamp("1900-01-01"),
            "end": pd.NaT,
            "source": "current_fallback",
        })

        return membership, "current_fallback_SURVIVORSHIP_BIASED"

    raise RuntimeError("No se pudo cargar ni reconstruir membership histórico.")


def active_tickers_on_date(membership: pd.DataFrame, date: pd.Timestamp) -> List[str]:
    """
    Devuelve los tickers activos en una fecha concreta según la tabla de pertenencia.
    """
    date = pd.Timestamp(date)

    active = membership[
        (membership["start"].isna() | (membership["start"] <= date)) &
        (membership["end"].isna() | (membership["end"] >= date))
    ]

    return sorted(active["yf_ticker"].dropna().unique().tolist())


def all_tickers_needed(membership: pd.DataFrame, start_date: str, end_date: str) -> List[str]:
    """
    Devuelve todos los tickers que podrían ser necesarios entre start_date y end_date.
    """
    start = pd.Timestamp(start_date)
    end = pd.Timestamp(end_date)

    relevant = membership[
        (membership["end"].isna() | (membership["end"] >= start)) &
        (membership["start"].isna() | (membership["start"] <= end))
    ]

    return sorted(relevant["yf_ticker"].dropna().unique().tolist())


In [ ]:
# ============================================================
# 4. DESCARGA Y PREPROCESADO DE DATOS


In [ ]:
# ============================================================

def download_adjusted_close(
    tickers: Iterable[str],
    start_date: str,
    end_date: str,
    chunk_size: int = 25,
    sleep_seconds: float = 1.0,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Descarga precios ajustados mediante yfinance.

    Con auto_adjust=True, la columna Close queda ajustada.
    Descarga por bloques para reducir errores en Colab/Yahoo.
    """
    tickers = sorted(set([to_yfinance_ticker(t) for t in tickers]))

    if not tickers:
        raise ValueError("No hay tickers para descargar.")

    all_prices = []
    report_rows = []

    print(f"Descargando {len(tickers)} tickers desde Yahoo Finance...")

    for i in range(0, len(tickers), chunk_size):
        chunk = tickers[i:i + chunk_size]
        print(f"  Bloque {i // chunk_size + 1}: {len(chunk)} tickers")

        try:
            data = yf.download(
                tickers=chunk,
                start=start_date,
                end=end_date,
                auto_adjust=True,
                progress=False,
                group_by="ticker",
                threads=True,
            )

        except Exception as e:
            for t in chunk:
                report_rows.append({
                    "yf_ticker": t,
                    "status": f"download_exception: {type(e).__name__}",
                    "coverage": 0.0,
                    "first_valid_date": None,
                    "last_valid_date": None,
                })

            time.sleep(sleep_seconds)
            continue

        prices_chunk = pd.DataFrame()

        if len(chunk) == 1:
            t = chunk[0]

            if isinstance(data, pd.DataFrame) and "Close" in data.columns:
                prices_chunk[t] = data["Close"]

        else:
            for t in chunk:
                try:
                    if isinstance(data.columns, pd.MultiIndex):
                        if t in data.columns.get_level_values(0):
                            close = data[t]["Close"].copy()
                            prices_chunk[t] = close
                    else:
                        if "Close" in data.columns:
                            prices_chunk[t] = data["Close"]

                except Exception:
                    pass

        if not prices_chunk.empty:
            prices_chunk = prices_chunk.sort_index()
            prices_chunk.index.name = "Date"
            prices_chunk = prices_chunk.dropna(axis=1, how="all")
            all_prices.append(prices_chunk)

        available = set(prices_chunk.columns) if not prices_chunk.empty else set()

        for t in chunk:
            if t in available:
                s = prices_chunk[t]
                coverage = float(s.notna().mean())
                first_date = s.first_valid_index()
                last_date = s.last_valid_index()
                status = "ok"
            else:
                coverage = 0.0
                first_date = None
                last_date = None
                status = "download_failed_or_no_close"

            report_rows.append({
                "yf_ticker": t,
                "status": status,
                "coverage": coverage,
                "first_valid_date": first_date,
                "last_valid_date": last_date,
            })

        time.sleep(sleep_seconds)

    if not all_prices:
        raise RuntimeError("No se han podido descargar precios. Revisa conexión, tickers o yfinance.")

    prices = pd.concat(all_prices, axis=1)
    prices = prices.loc[:, ~prices.columns.duplicated()]
    prices = prices.sort_index()
    prices.index.name = "Date"

    report = pd.DataFrame(report_rows)

    return prices, report


def clean_prices(
    prices: pd.DataFrame,
    min_coverage: float,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    """
    Limpieza conservadora.
    """
    prices = prices.copy()
    prices = prices[~prices.index.duplicated(keep="first")]
    prices = prices.sort_index()

    coverage = prices.notna().mean()

    keep = coverage[coverage >= min_coverage].index.tolist()
    excluded = coverage[coverage < min_coverage].index.tolist()

    exclusions = pd.DataFrame({
        "yf_ticker": excluded,
        "reason": f"coverage_below_{min_coverage:.0%}",
        "coverage": coverage.loc[excluded].values if len(excluded) > 0 else [],
    })

    clean = prices[keep].copy()
    clean = clean.ffill()

    return clean, exclusions


def compute_simple_returns(prices: pd.DataFrame) -> pd.DataFrame:
    returns = prices.pct_change(fill_method=None)
    returns = returns.replace([np.inf, -np.inf], np.nan)

    return returns


In [ ]:
# ============================================================
# 5. BACKTESTS BASE


In [ ]:
# ============================================================

def backtest_static_buy_hold(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Buy & Hold estático:
    - Toma los miembros activos al inicio del backtest.
    - Compra cartera equally weighted.
    - No rebalancea.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all")
    first_date = prices.index[0]

    active = active_tickers_on_date(membership, first_date)
    active = [t for t in active if t in prices.columns]

    first_prices = prices.loc[first_date, active].dropna()
    active = first_prices.index.tolist()

    if len(active) < config.min_assets_required:
        raise RuntimeError(
            f"Activos disponibles insuficientes para Buy & Hold: {len(active)} "
            f"(mínimo {config.min_assets_required})."
        )

    n = len(active)
    gross_weights = pd.Series(1.0 / n, index=active)

    entry_turnover = gross_weights.abs().sum()
    initial_cost = transaction_cost * entry_turnover * config.initial_capital
    investable_capital = config.initial_capital - initial_cost

    shares = (investable_capital * gross_weights) / first_prices

    portfolio_values = prices[active].ffill().mul(shares, axis=1).sum(axis=1)
    equity = portfolio_values.rename("strategy_equity")

    equity.iloc[0] = investable_capital

    returns = equity.pct_change(fill_method=None)
    returns.iloc[0] = equity.iloc[0] / config.initial_capital - 1.0
    returns = returns.rename("strategy_returns")

    weights = pd.DataFrame(0.0, index=prices.index, columns=active)
    weights.loc[:, active] = gross_weights.values

    metadata = {
        "strategy": "static_buy_hold",
        "first_date": str(first_date.date()),
        "n_assets": n,
        "transaction_cost": transaction_cost,
        "entry_turnover": float(entry_turnover),
        "initial_cost": float(initial_cost),
        "active_tickers": active,
    }

    return returns, equity, weights, metadata


def backtest_dynamic_equal_weight(
    prices: pd.DataFrame,
    membership: pd.DataFrame,
    config: BacktestConfig,
    transaction_cost: Optional[float] = None,
) -> Tuple[pd.Series, pd.Series, pd.DataFrame, dict]:
    """
    Equal-weight dinámico:
    - En cada fecha de rebalanceo selecciona miembros activos según membership histórico.
    - Rebalancea a pesos iguales.
    """
    transaction_cost = config.transaction_cost if transaction_cost is None else transaction_cost

    prices = prices.copy().dropna(axis=0, how="all").ffill()
    returns_assets = compute_simple_returns(prices).fillna(0.0)

    dates = prices.index

    rebalance_dates_raw = prices.resample(config.rebalance_frequency).last().index
    rebalance_dates = []

    for d in rebalance_dates_raw:
        idx = prices.index.get_indexer([d], method="pad")[0]

        if idx >= 0:
            rebalance_dates.append(prices.index[idx])

    rebalance_dates = sorted(set([d for d in rebalance_dates if d in dates]))

    all_cols = prices.columns.tolist()

    weights = pd.DataFrame(0.0, index=dates, columns=all_cols)

    current_weights = pd.Series(0.0, index=all_cols)
    prev_weights = pd.Series(0.0, index=all_cols)

    equity = pd.Series(index=dates, dtype=float, name="strategy_equity")
    strategy_returns = pd.Series(index=dates, dtype=float, name="strategy_returns")

    capital = config.initial_capital
    rebalance_count = 0
    total_turnover = 0.0

    for i, date in enumerate(dates):
        cost = 0.0

        if i == 0 or date in rebalance_dates:
            active = active_tickers_on_date(membership, date)
            active = [t for t in active if t in all_cols and not pd.isna(prices.loc[date, t])]

            if len(active) >= config.min_assets_required:
                target_weights = pd.Series(0.0, index=all_cols)
                target_weights.loc[active] = 1.0 / len(active)
            else:
                target_weights = prev_weights.copy()

            turnover = float((target_weights - prev_weights).abs().sum())
            cost = transaction_cost * turnover

            total_turnover += turnover
            rebalance_count += 1

            current_weights = target_weights.copy()
            prev_weights = current_weights.copy()

        if i == 0:
            r = -cost
        else:
            r_gross = float(current_weights.fillna(0.0).dot(returns_assets.iloc[i].fillna(0.0)))
            r = r_gross - cost

        capital *= (1.0 + r)

        equity.loc[date] = capital
        strategy_returns.loc[date] = r
        weights.loc[date] = current_weights.values

    metadata = {
        "strategy": "dynamic_equal_weight",
        "rebalance_frequency": config.rebalance_frequency,
        "transaction_cost": transaction_cost,
        "rebalance_count": rebalance_count,
        "total_turnover": total_turnover,
        "average_turnover_per_rebalance": total_turnover / rebalance_count if rebalance_count else np.nan,
    }

    return strategy_returns, equity, weights, metadata


def backtest_benchmark(
    benchmark_prices: pd.Series,
    config: BacktestConfig,
) -> Tuple[pd.Series, pd.Series]:
    """
    Benchmark Buy & Hold.
    No aplicamos costes al benchmark por defecto.
    """
    px = benchmark_prices.dropna().copy()
    px = px / px.iloc[0] * config.initial_capital
    px.name = "benchmark_equity"

    ret = px.pct_change(fill_method=None)
    ret.iloc[0] = 0.0
    ret.name = "benchmark_returns"

    return ret, px


In [ ]:
# ============================================================
# 6. MÉTRICAS


In [ ]:
# ============================================================

def format_index_endpoint(x):
    """
    Convierte un índice a string de forma robusta.
    """
    if hasattr(x, "date"):
        return str(x.date())

    return str(x)


def max_drawdown(equity: pd.Series) -> float:
    equity = equity.dropna()

    if len(equity) == 0:
        return np.nan

    running_max = equity.cummax()
    dd = equity / running_max - 1.0

    return float(dd.min())


def drawdown_series(equity: pd.Series) -> pd.Series:
    equity = equity.dropna()

    if len(equity) == 0:
        return pd.Series(dtype=float, name="drawdown")

    running_max = equity.cummax()

    return (equity / running_max - 1.0).rename("drawdown")


def annualized_volatility(
    returns: pd.Series,
    trading_days: int = 252,
) -> float:
    returns = returns.dropna()

    if len(returns) < 2:
        return np.nan

    return float(returns.std(ddof=1) * math.sqrt(trading_days))


def cagr_from_equity(
    equity: pd.Series,
    initial_capital: float,
    trading_days: int = 252,
) -> float:
    equity = equity.dropna()

    if len(equity) < 2:
        return np.nan

    years = len(equity) / trading_days
    final_value = equity.iloc[-1]

    if final_value <= 0:
        return -1.0

    return float((final_value / initial_capital) ** (1.0 / years) - 1.0)


def compute_metrics(
    returns: pd.Series,
    equity: pd.Series,
    config: BacktestConfig,
    label: str,
) -> pd.Series:
    returns = returns.dropna()
    equity = equity.dropna()

    if len(returns) == 0 or len(equity) == 0:
        return pd.Series({
            "label": label,
            "start": None,
            "end": None,
            "n_observations": 0,
            "cumulative_return": np.nan,
            "CAGR": np.nan,
            "annualized_volatility": np.nan,
            "Sharpe": np.nan,
            "Sortino": np.nan,
            "max_drawdown": np.nan,
            "Calmar": np.nan,
            "hit_ratio": np.nan,
            "risk_free_rate_annual": config.risk_free_rate_annual,
        })

    rf_daily = (1.0 + config.risk_free_rate_annual) ** (1.0 / config.trading_days_per_year) - 1.0
    excess = returns - rf_daily

    cumulative_return = float(equity.iloc[-1] / config.initial_capital - 1.0)

    cagr = cagr_from_equity(
        equity,
        config.initial_capital,
        config.trading_days_per_year,
    )

    vol = annualized_volatility(
        returns,
        config.trading_days_per_year,
    )

    if len(excess) > 1 and excess.std(ddof=1) > 0:
        sharpe = float(
            excess.mean() / excess.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sharpe = np.nan

    downside = excess[excess < 0]

    if len(downside) > 1 and downside.std(ddof=1) > 0:
        sortino = float(
            excess.mean() / downside.std(ddof=1) * math.sqrt(config.trading_days_per_year)
        )
    else:
        sortino = np.nan

    mdd = max_drawdown(equity)

    if not np.isnan(mdd) and mdd < 0:
        calmar = float(cagr / abs(mdd))
    else:
        calmar = np.nan

    hit_ratio = float((returns > 0).mean())

    return pd.Series({
        "label": label,
        "start": format_index_endpoint(equity.index[0]),
        "end": format_index_endpoint(equity.index[-1]),
        "n_observations": int(len(returns)),
        "cumulative_return": cumulative_return,
        "CAGR": cagr,
        "annualized_volatility": vol,
        "Sharpe": sharpe,
        "Sortino": sortino,
        "max_drawdown": mdd,
        "Calmar": calmar,
        "hit_ratio": hit_ratio,
        "risk_free_rate_annual": config.risk_free_rate_annual,
    })


def compute_turnover(weights: pd.DataFrame) -> pd.Series:
    if weights.empty:
        return pd.Series(dtype=float, name="turnover")

    turnover = weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = weights.iloc[0].abs().sum()

    return turnover.rename("turnover")


In [ ]:
# ============================================================
# 7. BOOTSTRAP SIMPLE


In [ ]:
# ============================================================

def bootstrap_metrics_simple(
    returns: pd.Series,
    config: BacktestConfig,
    n_bootstrap: int = 1000,
) -> pd.DataFrame:
    """
    Bootstrap simple sobre rentabilidades.
    Más adelante podemos sustituirlo por block bootstrap.
    """
    rng = np.random.default_rng(config.random_seed)
    r = returns.dropna().values

    rows = []

    for b in range(n_bootstrap):
        sample = rng.choice(r, size=len(r), replace=True)
        equity = pd.Series(config.initial_capital * np.cumprod(1.0 + sample))
        sample_returns = pd.Series(sample)

        m = compute_metrics(sample_returns, equity, config, label=f"bootstrap_{b}")
        rows.append(m)

    return pd.DataFrame(rows)


In [ ]:
# ============================================================
# 8. FIGURAS


In [ ]:
# ============================================================

def plot_equity_curves(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    strategy_equity.dropna().plot(label="Strategy")
    benchmark_equity.dropna().plot(label="Benchmark")
    plt.title("Curva de capital")
    plt.xlabel("Fecha")
    plt.ylabel("Capital normalizado")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


def plot_drawdowns(
    strategy_equity: pd.Series,
    benchmark_equity: pd.Series,
    output_path: str | Path,
) -> None:
    plt.figure(figsize=(10, 6))
    drawdown_series(strategy_equity).plot(label="Strategy")
    drawdown_series(benchmark_equity).plot(label="Benchmark")
    plt.title("Drawdown")
    plt.xlabel("Fecha")
    plt.ylabel("Drawdown")
    plt.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=160)
    plt.close()


In [ ]:
# ============================================================
# 9. PIPELINE PRINCIPAL


In [ ]:
# ============================================================

def run_block0(config: BacktestConfig = CONFIG) -> None:
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    save_json(run_metadata, out / "config_run.json")

    # Universo histórico
    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # Descarga
    prices_raw, download_report = download_adjusted_close(
        all_download_tickers,
        config.start_date,
        config.end_date,
    )

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # Backtest de estrategia
    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # Benchmark
    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    # Alinear fechas
    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # Métricas principales
    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # Turnover
    turnover = compute_turnover(weights)

    # Sensibilidad a costes
    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # Bootstrap simple
    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # Exportar outputs
    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # Metadata
    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # Figuras
    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # Resumen por pantalla
    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 0 cargado correctamente. Ahora cambia el Bloque 3 y ejecuta el Bloque 4.")


In [ ]:
# ============================================================
# BLOQUE 2.5 - CACHÉ DE MEMBERSHIP Y DATOS DE MERCADO


In [ ]:
# ============================================================

import hashlib


def make_market_data_cache_key(
    tickers: list,
    start_date: str,
    end_date: str,
    data_source: str,
) -> str:
    """
    Genera una clave única para identificar una descarga de datos.

    Si cambias:
    - tickers,
    - start_date,
    - end_date,
    - fuente de datos,

    se genera una caché distinta.
    """
    raw = {
        "tickers": sorted(tickers),
        "start_date": start_date,
        "end_date": end_date,
        "data_source": data_source,
    }

    raw_string = json.dumps(raw, sort_keys=True)
    return hashlib.md5(raw_string.encode("utf-8")).hexdigest()[:12]


def load_or_download_market_data(
    tickers: list,
    config: BacktestConfig,
) -> Tuple[pd.DataFrame, pd.DataFrame, str]:
    """
    Carga precios desde caché si existen.

    Si no existen o si CONFIG.force_redownload_market_data=True,
    descarga desde Yahoo Finance y guarda caché.
    """
    cache_dir = ensure_dir(getattr(config, "market_data_cache_dir", "data/cache"))

    cache_key = make_market_data_cache_key(
        tickers=tickers,
        start_date=config.start_date,
        end_date=config.end_date,
        data_source=config.data_source,
    )

    prices_cache_path = cache_dir / f"prices_{cache_key}.csv"
    report_cache_path = cache_dir / f"download_report_{cache_key}.csv"
    metadata_cache_path = cache_dir / f"market_data_metadata_{cache_key}.json"

    use_cached_market_data = getattr(config, "use_cached_market_data", True)
    force_redownload_market_data = getattr(config, "force_redownload_market_data", False)

    if (
        use_cached_market_data
        and not force_redownload_market_data
        and prices_cache_path.exists()
        and report_cache_path.exists()
    ):
        print(f"Cargando datos de mercado desde caché: {prices_cache_path}")

        prices = pd.read_csv(
            prices_cache_path,
            index_col=0,
            parse_dates=True,
        )

        prices.index.name = "Date"
        prices = prices.sort_index()

        report = pd.read_csv(report_cache_path)

        return prices, report, "market_data_cache"

    print("No hay caché válida de datos de mercado. Descargando desde Yahoo Finance...")

    prices, report = download_adjusted_close(
        tickers,
        config.start_date,
        config.end_date,
    )

    prices.to_csv(prices_cache_path)
    report.to_csv(report_cache_path, index=False)

    metadata = {
        "created_at_utc": now_utc_iso(),
        "cache_key": cache_key,
        "start_date": config.start_date,
        "end_date": config.end_date,
        "data_source": config.data_source,
        "n_tickers_requested": len(tickers),
        "tickers": sorted(tickers),
        "prices_cache_path": str(prices_cache_path),
        "report_cache_path": str(report_cache_path),
    }

    save_json(metadata, metadata_cache_path)

    print(f"Datos guardados en caché: {prices_cache_path}")

    return prices, report, "market_data_downloaded"


def run_block0(config: BacktestConfig = CONFIG) -> None:
    """
    Pipeline principal con caché.

    Cambios respecto al anterior:
    - Puede reutilizar membership histórico reconstruido.
    - Puede reutilizar precios descargados de Yahoo Finance.
    - Evita repetir los 6 minutos de descarga/reconstrucción cada vez.
    """
    warnings.filterwarnings("ignore", category=FutureWarning)
    np.random.seed(config.random_seed)

    out = ensure_dir(config.output_dir)
    ensure_dir(out / "figures")
    ensure_dir("data/raw")
    ensure_dir("data/processed")
    ensure_dir("logs")
    ensure_dir("data/cache")

    run_metadata = {
        "run_timestamp_utc": now_utc_iso(),
        "config": asdict(config),
    }

    # Guardamos también atributos añadidos dinámicamente en Colab.
    dynamic_config_attrs = [
        "use_cached_market_data",
        "force_redownload_market_data",
        "market_data_cache_dir",
    ]

    for attr in dynamic_config_attrs:
        if hasattr(config, attr):
            run_metadata["config"][attr] = getattr(config, attr)

    save_json(run_metadata, out / "config_run.json")

    # ========================================================
    # 1. Universo histórico
    # ========================================================

    membership, membership_source = load_membership_schedule(config)
    run_metadata["membership_source"] = membership_source

    tickers_needed = all_tickers_needed(
        membership,
        config.start_date,
        config.end_date,
    )

    all_download_tickers = sorted(set(tickers_needed + [config.benchmark_ticker]))

    # ========================================================
    # 2. Datos de mercado con caché
    # ========================================================

    prices_raw, download_report, market_data_source = load_or_download_market_data(
        all_download_tickers,
        config,
    )

    run_metadata["market_data_source"] = market_data_source

    prices_raw.to_csv("data/raw/prices_raw.csv")
    download_report.to_csv("logs/download_report.csv", index=False)

    if config.benchmark_ticker not in prices_raw.columns:
        raise RuntimeError(
            f"No se pudo descargar/cargar el benchmark {config.benchmark_ticker}. "
            f"Prueba con '^GSPC' o revisa conexión/yfinance."
        )

    benchmark_prices = prices_raw[config.benchmark_ticker].dropna()

    asset_prices_raw = prices_raw.drop(columns=[config.benchmark_ticker], errors="ignore")
    asset_prices, exclusions = clean_prices(asset_prices_raw, config.min_price_coverage)

    asset_prices.to_csv("data/processed/prices_clean.csv")
    compute_simple_returns(asset_prices).to_csv("data/processed/returns.csv")
    exclusions.to_csv("logs/exclusions.csv", index=False)

    # ========================================================
    # 3. Backtest de estrategia
    # ========================================================

    if config.strategy_mode == "static_buy_hold":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_static_buy_hold(
            asset_prices,
            membership,
            config,
        )

    elif config.strategy_mode == "dynamic_equal_weight":
        strategy_returns, strategy_equity, weights, strategy_meta = backtest_dynamic_equal_weight(
            asset_prices,
            membership,
            config,
        )

    else:
        raise ValueError(f"strategy_mode no reconocido: {config.strategy_mode}")

    # ========================================================
    # 4. Benchmark
    # ========================================================

    benchmark_returns, benchmark_equity = backtest_benchmark(
        benchmark_prices,
        config,
    )

    common_dates = strategy_equity.index.intersection(benchmark_equity.index)

    strategy_equity = strategy_equity.loc[common_dates]
    strategy_returns = strategy_returns.loc[common_dates]

    benchmark_equity = benchmark_equity.loc[common_dates]
    benchmark_returns = benchmark_returns.loc[common_dates]

    # ========================================================
    # 5. Métricas principales
    # ========================================================

    strategy_metrics = compute_metrics(
        strategy_returns,
        strategy_equity,
        config,
        "SP100_strategy",
    )

    benchmark_metrics = compute_metrics(
        benchmark_returns,
        benchmark_equity,
        config,
        config.benchmark_name,
    )

    metrics = pd.DataFrame([strategy_metrics, benchmark_metrics])

    # ========================================================
    # 6. Turnover
    # ========================================================

    turnover = compute_turnover(weights)

    # ========================================================
    # 7. Sensibilidad a costes
    # ========================================================

    sensitivity_rows = []

    for cost in config.cost_sensitivity:
        if config.strategy_mode == "static_buy_hold":
            r_s, eq_s, _, _ = backtest_static_buy_hold(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        else:
            r_s, eq_s, _, _ = backtest_dynamic_equal_weight(
                asset_prices,
                membership,
                config,
                transaction_cost=cost,
            )

        eq_s = eq_s.loc[eq_s.index.intersection(common_dates)]
        r_s = r_s.loc[eq_s.index]

        m = compute_metrics(r_s, eq_s, config, f"strategy_cost_{cost}")
        m["transaction_cost"] = cost
        sensitivity_rows.append(m)

    sensitivity = pd.DataFrame(sensitivity_rows)

    # ========================================================
    # 8. Bootstrap simple
    # ========================================================

    boot = bootstrap_metrics_simple(
        strategy_returns,
        config,
        n_bootstrap=1000,
    )

    boot_summary = boot[["CAGR", "Sharpe", "Sortino", "max_drawdown", "Calmar"]].quantile(
        [0.025, 0.05, 0.50, 0.95, 0.975]
    )

    # ========================================================
    # 9. Exportar outputs
    # ========================================================

    strategy_returns.to_csv(out / "strategy_returns.csv")
    strategy_equity.to_csv(out / "strategy_equity.csv")

    benchmark_returns.to_csv(out / "benchmark_returns.csv")
    benchmark_equity.to_csv(out / "benchmark_equity.csv")

    weights.to_csv(out / "weights.csv")
    turnover.to_csv(out / "turnover.csv")

    metrics.to_csv(out / "metrics.csv", index=False)
    sensitivity.to_csv(out / "cost_sensitivity.csv", index=False)

    boot.to_csv(out / "bootstrap_metrics.csv", index=False)
    boot_summary.to_csv(out / "bootstrap_summary.csv")

    # ========================================================
    # 10. Metadata
    # ========================================================

    run_metadata["strategy_metadata"] = strategy_meta
    run_metadata["n_raw_assets_downloaded"] = len(tickers_needed)
    run_metadata["n_assets_after_cleaning"] = int(asset_prices.shape[1])
    run_metadata["n_excluded_assets"] = int(len(exclusions))
    run_metadata["benchmark_ticker"] = config.benchmark_ticker

    if "SURVIVORSHIP_BIASED" in membership_source:
        membership_warning = "WARNING: current/fallback constituents used. Survivorship bias present."
    elif "wikipedia" in membership_source.lower():
        membership_warning = (
            "Historical public reconstruction from Wikipedia page revisions. "
            "This avoids using future constituents, but is not the official licensed S&P DJI constituent history."
        )
    else:
        membership_warning = "Historical membership file used."

    run_metadata["membership_warning"] = membership_warning

    save_json(run_metadata, out / "metadata.json")

    # ========================================================
    # 11. Figuras
    # ========================================================

    plot_equity_curves(
        strategy_equity,
        benchmark_equity,
        out / "figures/equity_curves.png",
    )

    plot_drawdowns(
        strategy_equity,
        benchmark_equity,
        out / "figures/drawdowns.png",
    )

    # ========================================================
    # 12. Resumen por pantalla
    # ========================================================

    print("\n=== Bloque 0 completado ===")
    print(f"Modo estrategia: {config.strategy_mode}")
    print(f"Fuente membership: {membership_source}")
    print(f"Fuente datos mercado: {market_data_source}")
    print(f"Activos tras limpieza: {asset_prices.shape[1]}")
    print(f"Benchmark: {config.benchmark_ticker}")
    print(f"Outputs guardados en: {out.resolve()}")

    print("\nMétricas principales:")
    with pd.option_context("display.max_columns", None, "display.width", 160):
        print(metrics)


print("Bloque 2.5 cargado correctamente. run_block0 ahora usa caché de datos.")

"""Configuración"""


In [ ]:
# ============================================================
# cELDA 3 - CONFIGURACIÓN DEL EXPERIMENTO


In [ ]:
# ============================================================

from pathlib import Path

# Asegurar carpetas en Drive
Path(f"{TFG_BASE_DIR}").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/data/cache").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/results").mkdir(parents=True, exist_ok=True)
Path(f"{TFG_BASE_DIR}/logs").mkdir(parents=True, exist_ok=True)

CONFIG.start_date = "2021-01-01"
CONFIG.end_date = "2026-01-01"

# Benchmark S&P 500.
CONFIG.benchmark_ticker = "SPY"

# Usamos estrategia dinámica para respetar entradas/salidas del universo histórico.
CONFIG.strategy_mode = "dynamic_equal_weight"
CONFIG.rebalance_frequency = "M"

# Costes
CONFIG.transaction_cost = 0.001

# Tasa libre de riesgo anual provisional.
CONFIG.risk_free_rate_annual = 0.0225

# Reconstrucción histórica del S&P 100.
CONFIG.use_reconstructed_membership = True

# False = no reconstruye desde Wikipedia si ya existe el CSV reconstruido.
CONFIG.force_rebuild_membership = False

CONFIG.membership_history_frequency = "MS"

# Si falla la reconstrucción histórica, NO vuelve a lista actual.
CONFIG.allow_current_fallback = False

# Caché de precios.
CONFIG.use_cached_market_data = True

# False = no vuelve a descargar precios si ya existen en caché.
CONFIG.force_redownload_market_data = False

# Guardamos caché en Drive.
CONFIG.market_data_cache_dir = f"{TFG_BASE_DIR}/data/cache"

# Guardamos resultados en Drive.
CONFIG.output_dir = f"{TFG_BASE_DIR}/results/block0_sp100_survivorship_corrected"

# Guardamos membership histórico y logs en Drive.
CONFIG.reconstructed_membership_file = f"{TFG_BASE_DIR}/data/sp100_membership_reconstructed_wikipedia.csv"
CONFIG.snapshot_log_file = f"{TFG_BASE_DIR}/data/sp100_wikipedia_snapshot_log.csv"

CONFIG

"""Ejecutar"""

run_block0(CONFIG)

"""Resultados"""


In [ ]:
# ============================================================
# CELDA 5 - VER RESULTADOS PRINCIPALES


In [ ]:
# ============================================================

from pathlib import Path
import pandas as pd
import json

results_dir = Path(CONFIG.output_dir)

metrics_path = results_dir / "metrics.csv"
sensitivity_path = results_dir / "cost_sensitivity.csv"
bootstrap_summary_path = results_dir / "bootstrap_summary.csv"
metadata_path = results_dir / "metadata.json"

print(f"Carpeta de resultados: {results_dir}")

if not metrics_path.exists():
    raise FileNotFoundError(
        f"No existe {metrics_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

metrics = pd.read_csv(metrics_path)
cost_sensitivity = pd.read_csv(sensitivity_path)
bootstrap_summary = pd.read_csv(bootstrap_summary_path, index_col=0)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("\n=== Metadata clave ===")
print("Fuente membership:", metadata.get("membership_source"))
print("Aviso membership:", metadata.get("membership_warning"))
print("Fuente datos mercado:", metadata.get("market_data_source"))
print("Benchmark:", metadata.get("benchmark_ticker"))
print("Activos tras limpieza:", metadata.get("n_assets_after_cleaning"))

print("\n=== Métricas principales ===")
display(metrics)

print("\n=== Sensibilidad a costes ===")
display(cost_sensitivity)

print("\n=== Bootstrap summary ===")
display(bootstrap_summary)

"""Figuras"""


In [ ]:
# ============================================================
# CELDA 6 - VER FIGURAS


In [ ]:
# ============================================================

from pathlib import Path
from IPython.display import Image, display

results_dir = Path(CONFIG.output_dir)
figures_dir = results_dir / "figures"

equity_path = figures_dir / "equity_curves.png"
drawdown_path = figures_dir / "drawdowns.png"

print(f"Carpeta de figuras: {figures_dir}")

if not equity_path.exists():
    raise FileNotFoundError(
        f"No existe {equity_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

if not drawdown_path.exists():
    raise FileNotFoundError(
        f"No existe {drawdown_path}. Ejecuta primero el Bloque 4: run_block0(CONFIG)"
    )

print("\n=== Curva de capital ===")
display(Image(filename=str(equity_path)))

print("\n=== Drawdown ===")
display(Image(filename=str(drawdown_path)))

"""Descargar"""

!zip -r block0_results.zip results data logs > /dev/null

from google.colab import files
files.download("block0_results.zip")


In [ ]:
# ============================================================
# Celda C4.1 - Instalación, imports y configuración


In [ ]:
# ============================================================

!pip -q install scipy statsmodels

from pathlib import Path
from io import BytesIO, StringIO
import zipfile
import urllib.request
import json
import math
import warnings
import copy

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import statsmodels.api as sm

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# Fechas C4
# ------------------------------------------------------------

# Usamos datos previos solo para estimar la primera cartera.
C4_DATA_START = "2020-01-01"

# Periodo real de evaluación.
C4_ANALYSIS_START = "2021-01-01"
C4_ANALYSIS_END = "2025-12-31"

# Fecha final de descarga.
C4_END = "2026-01-01"

# ------------------------------------------------------------
# Carpeta de salida
# ------------------------------------------------------------

C4_OUTPUT_DIR = Path(f"{TFG_BASE_DIR}/results/C4_markowitz_cml_final_2021_2025")
C4_TABLES_DIR = C4_OUTPUT_DIR / "tables"
C4_FIGURES_DIR = C4_OUTPUT_DIR / "figures"
C4_DATA_DIR = C4_OUTPUT_DIR / "data"

for p in [C4_OUTPUT_DIR, C4_TABLES_DIR, C4_FIGURES_DIR, C4_DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Configuración general
# ------------------------------------------------------------

C4_RANDOM_SEED = CONFIG.random_seed
C4_TRANSACTION_COST = CONFIG.transaction_cost
C4_RISK_FREE_RATE_ANNUAL = CONFIG.risk_free_rate_annual
C4_RISK_FREE_DAILY = (1.0 + C4_RISK_FREE_RATE_ANNUAL) ** (1.0 / CONFIG.trading_days_per_year) - 1.0

# Ventana de estimación y frecuencia de rebalanceo.
C4_LOOKBACK_DAYS = 126
C4_REBALANCE_FREQUENCY = "M"

# Restricción máxima por activo.
C4_MAX_WEIGHT = 0.30

# Capital Market Line:
# En la ejecución final usamos L=1, es decir, sin apalancamiento.
C4_CML_LEVERAGE = 1.00

# Sin coste de financiación adicional porque no hay apalancamiento.
C4_BORROW_SPREAD_ANNUAL = 0.00

# Bootstrap.
C4_N_BOOTSTRAP = 1000
C4_BLOCK_LENGTH = 20

rng = np.random.default_rng(C4_RANDOM_SEED)

print("C4_OUTPUT_DIR:", C4_OUTPUT_DIR)
print("Datos desde:", C4_DATA_START)
print("Periodo de análisis:", C4_ANALYSIS_START, "->", C4_ANALYSIS_END)
print("Coste transacción:", C4_TRANSACTION_COST)
print("Lookback:", C4_LOOKBACK_DAYS)
print("Frecuencia rebalanceo:", C4_REBALANCE_FREQUENCY)
print("Peso máximo:", C4_MAX_WEIGHT)
print("CML leverage:", C4_CML_LEVERAGE)


In [ ]:
# ============================================================
# Celda C4.2 - Cargar datos desde la infraestructura común C0


In [ ]:
# ============================================================

# ------------------------------------------------------------
# Membership histórica
# ------------------------------------------------------------

C4_MEMBERSHIP_CONFIG = copy.deepcopy(CONFIG)

C4_MEMBERSHIP_CONFIG.start_date = C4_ANALYSIS_START
C4_MEMBERSHIP_CONFIG.end_date = C4_END

# Reutilizamos la membership histórica ya cacheada en C0.
# No forzamos reconstrucción desde Wikipedia.
C4_MEMBERSHIP_CONFIG.use_reconstructed_membership = True
C4_MEMBERSHIP_CONFIG.force_rebuild_membership = False
C4_MEMBERSHIP_CONFIG.allow_current_fallback = False
C4_MEMBERSHIP_CONFIG.membership_history_frequency = "MS"

membership, membership_source = load_membership_schedule(C4_MEMBERSHIP_CONFIG)

try:
    tickers_needed = all_tickers_needed(
        membership,
        C4_ANALYSIS_START,
        C4_END,
    )
except TypeError:
    tickers_needed = all_tickers_needed(
        membership,
        C4_MEMBERSHIP_CONFIG,
    )

benchmark_yf = to_yfinance_ticker(CONFIG.benchmark_ticker)

all_download_tickers = sorted(set(tickers_needed + [benchmark_yf]))

# ------------------------------------------------------------
# Datos de mercado
# ------------------------------------------------------------

C4_DATA_CONFIG = copy.deepcopy(CONFIG)
C4_DATA_CONFIG.start_date = C4_DATA_START
C4_DATA_CONFIG.end_date = C4_END
C4_DATA_CONFIG.benchmark_ticker = benchmark_yf
C4_DATA_CONFIG.transaction_cost = C4_TRANSACTION_COST
C4_DATA_CONFIG.risk_free_rate_annual = C4_RISK_FREE_RATE_ANNUAL
C4_DATA_CONFIG.output_dir = str(C4_OUTPUT_DIR)

# Reutilizamos caché de precios.
C4_DATA_CONFIG.market_data_cache_dir = f"{TFG_BASE_DIR}/data/cache"
C4_DATA_CONFIG.use_cached_market_data = True
C4_DATA_CONFIG.force_redownload_market_data = False

prices_raw, download_report, market_data_source = load_or_download_market_data(
    all_download_tickers,
    C4_DATA_CONFIG,
)

prices_raw = prices_raw.sort_index()
prices_raw = prices_raw[~prices_raw.index.duplicated(keep="first")]

if benchmark_yf not in prices_raw.columns:
    raise RuntimeError(f"No se encuentra el benchmark {benchmark_yf} en prices_raw.")

benchmark_prices = prices_raw[benchmark_yf].dropna().copy()
asset_prices_raw = prices_raw.drop(columns=[benchmark_yf], errors="ignore").copy()

# ------------------------------------------------------------
# Limpieza por cobertura SOLO en periodo de análisis
# ------------------------------------------------------------

analysis_slice = asset_prices_raw.loc[C4_ANALYSIS_START:C4_ANALYSIS_END].copy()
coverage_analysis = analysis_slice.notna().mean()

keep_cols = coverage_analysis[
    coverage_analysis >= C4_DATA_CONFIG.min_price_coverage
].index.tolist()

excluded_cols = coverage_analysis[
    coverage_analysis < C4_DATA_CONFIG.min_price_coverage
].index.tolist()

prices_clean = asset_prices_raw[keep_cols].copy()
prices_clean = prices_clean.ffill()

asset_returns = compute_simple_returns(prices_clean).fillna(0.0)

benchmark_returns = benchmark_prices.pct_change(fill_method=None)
benchmark_returns = benchmark_returns.replace([np.inf, -np.inf], np.nan).fillna(0.0)
benchmark_returns.name = "Benchmark_SPY"

# Alineamos benchmark y activos.
common_dates = prices_clean.index.intersection(benchmark_returns.index)

prices_clean = prices_clean.loc[common_dates].ffill()
asset_returns = asset_returns.loc[common_dates].fillna(0.0)
benchmark_returns = benchmark_returns.loc[common_dates].fillna(0.0)
benchmark_equity = c4_equity_from_returns(benchmark_returns) if "c4_equity_from_returns" in globals() else (1.0 + benchmark_returns).cumprod()

# ------------------------------------------------------------
# Guardar diagnósticos
# ------------------------------------------------------------

download_report.to_csv(C4_DATA_DIR / "C4_download_report.csv", index=False)

exclusions = pd.DataFrame({
    "yf_ticker": excluded_cols,
    "reason": f"coverage_analysis_below_{C4_DATA_CONFIG.min_price_coverage:.0%}",
})

if len(excluded_cols) > 0:
    exclusions["coverage_analysis"] = coverage_analysis.loc[excluded_cols].values

exclusions.to_csv(C4_TABLES_DIR / "C4_exclusions.csv", index=False)

prices_clean.to_csv(C4_DATA_DIR / "C4_prices_clean.csv")
asset_returns.to_csv(C4_DATA_DIR / "C4_asset_returns.csv")
benchmark_returns.to_csv(C4_DATA_DIR / "C4_benchmark_returns.csv")

print("Membership source:", membership_source)
print("Market data source:", market_data_source)
print("Activos válidos:", prices_clean.shape[1])
print("Activos excluidos:", len(excluded_cols))
print("Observaciones totales:", prices_clean.shape[0])
print("Fechas datos:", prices_clean.index.min(), "->", prices_clean.index.max())
print("Periodo análisis:", C4_ANALYSIS_START, "->", C4_ANALYSIS_END)

display(prices_clean.head())
display(prices_clean.tail())


In [ ]:
# ============================================================
# Celda C4.3 - Descargar factores FF5 + MOM de Kenneth French


In [ ]:
# ============================================================

FF5_DAILY_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip"
MOM_DAILY_URL = "https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Momentum_Factor_daily_CSV.zip"


def c4_download_zip_text(url):
    with urllib.request.urlopen(url) as response:
        raw = response.read()

    z = zipfile.ZipFile(BytesIO(raw))
    csv_names = [name for name in z.namelist() if name.lower().endswith(".csv")]

    if not csv_names:
        raise ValueError(f"No se encontró CSV dentro del zip: {url}")

    with z.open(csv_names[0]) as f:
        text = f.read().decode("utf-8", errors="replace")

    return text


def c4_parse_french_csv(text, expected_header_fragment):
    lines = text.splitlines()

    start_idx = None
    for i, line in enumerate(lines):
        clean = line.replace(" ", "")
        if expected_header_fragment.replace(" ", "") in clean:
            start_idx = i
            break

    if start_idx is None:
        raise ValueError(f"No se encontró cabecera con: {expected_header_fragment}")

    end_idx = None
    for j in range(start_idx + 1, len(lines)):
        first = lines[j].split(",")[0].strip()
        if first == "" or not first.isdigit():
            end_idx = j
            break

    if end_idx is None:
        end_idx = len(lines)

    csv_text = "\n".join(lines[start_idx:end_idx])
    df = pd.read_csv(StringIO(csv_text))

    first_col = df.columns[0]
    df = df.rename(columns={first_col: "Date"})
    df["Date"] = pd.to_datetime(df["Date"].astype(str), format="%Y%m%d", errors="coerce")
    df = df.dropna(subset=["Date"]).set_index("Date").sort_index()

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # Los factores vienen en porcentaje diario, no en tanto por uno.
    return df


ff5_text = c4_download_zip_text(FF5_DAILY_URL)
mom_text = c4_download_zip_text(MOM_DAILY_URL)

ff5 = c4_parse_french_csv(ff5_text, "Mkt-RF")
mom = c4_parse_french_csv(mom_text, "Mom")

# Normalizar nombre de momentum.
mom.columns = [str(c).strip() for c in mom.columns]

if "Mom" not in mom.columns:
    mom = mom.rename(columns={mom.columns[0]: "Mom"})

factors = ff5.join(mom[["Mom"]], how="inner")

factors = factors.rename(columns={
    "Mkt-RF": "Mkt_RF",
    "Mom": "MOM",
})

# Nos quedamos con el rango del experimento.
factors = factors.loc[
    (factors.index >= pd.Timestamp(CONFIG.start_date)) &
    (factors.index < pd.Timestamp(CONFIG.end_date))
].copy()

factors.to_csv(C4_DATA_DIR / "C4_ff5_mom_daily_factors.csv")

print("Factores descargados:", factors.shape)
display(factors.head())
display(factors.tail())


In [ ]:
# ============================================================
# Celda C4.3b - Configuración definitiva Markowitz-CML


In [ ]:
# ============================================================

C4_OUTPUT_DIR = Path(f"{TFG_BASE_DIR}/results/C4_markowitz_cml_final_2021_2025")
C4_TABLES_DIR = C4_OUTPUT_DIR / "tables"
C4_FIGURES_DIR = C4_OUTPUT_DIR / "figures"
C4_DATA_DIR = C4_OUTPUT_DIR / "data"

for p in [C4_OUTPUT_DIR, C4_TABLES_DIR, C4_FIGURES_DIR, C4_DATA_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# Ventana de estimación.
C4_LOOKBACK_DAYS = 126

# Restricción máxima por activo para evitar concentración extrema.
C4_MAX_WEIGHT = 0.30

# En la ejecución final no usamos apalancamiento.
C4_CML_LEVERAGE = 1.00

# Sin coste de financiación adicional.
C4_BORROW_SPREAD_ANNUAL = 0.00

# Bootstrap.
C4_N_BOOTSTRAP = 1000
C4_BLOCK_LENGTH = 20

print("Configuración C4 definitiva:")
print("Output:", C4_OUTPUT_DIR)
print("Datos desde:", C4_DATA_START)
print("Periodo análisis:", C4_ANALYSIS_START, "->", C4_ANALYSIS_END)
print("Lookback:", C4_LOOKBACK_DAYS)
print("Max weight:", C4_MAX_WEIGHT)
print("CML leverage:", C4_CML_LEVERAGE)
print("Borrow spread annual:", C4_BORROW_SPREAD_ANNUAL)
print("Bootstrap:", C4_N_BOOTSTRAP)
print("Block length:", C4_BLOCK_LENGTH)


In [ ]:
# ============================================================
# Celda C4.4 - Funciones Markowitz-CML


In [ ]:
# ============================================================

def c4_equity_from_returns(returns, initial_capital=1.0):
    r = pd.Series(returns).fillna(0.0)
    eq = initial_capital * (1.0 + r).cumprod()
    eq.name = "equity"
    return eq


def c4_drawdown_series(equity):
    eq = pd.Series(equity).dropna()
    return eq / eq.cummax() - 1.0


def c4_compute_metrics(returns, label):
    returns = pd.Series(returns).dropna()
    equity = c4_equity_from_returns(returns, CONFIG.initial_capital)
    return compute_metrics(returns, equity, CONFIG, label)


def c4_get_rebalance_dates(index, frequency="M", min_start_idx=0):
    dates_raw = pd.Series(data=np.arange(len(index)), index=index).resample(frequency).last().index
    rebalance_dates = []

    for d in dates_raw:
        idx = index.get_indexer([d], method="pad")[0]
        if idx >= min_start_idx:
            rebalance_dates.append(index[idx])

    return sorted(set([d for d in rebalance_dates if d in index]))


def c4_active_assets_on_date(date, columns, min_required=CONFIG.min_assets_required):
    active = active_tickers_on_date(membership, date)
    active = [t for t in active if t in columns]

    if len(active) < min_required:
        return []

    return active


def c4_prepare_estimation_window(date, active_assets):
    idx = prices_clean.index.get_loc(date)

    if idx < C4_LOOKBACK_DAYS:
        return None, None, None

    window_returns = asset_returns.iloc[idx - C4_LOOKBACK_DAYS:idx][active_assets].copy()
    window_returns = window_returns.dropna(axis=1, how="any")

    if window_returns.shape[1] < CONFIG.min_assets_required:
        return None, None, None

    cols = window_returns.columns.tolist()

    mu = window_returns.mean().values
    cov = window_returns.cov().values

    # Regularización pequeña para evitar problemas numéricos.
    cov = cov + np.eye(cov.shape[0]) * 1e-6

    return cols, mu, cov


def c4_equal_weights(n):
    if n <= 0:
        return np.array([])
    return np.ones(n) / n


def c4_optimize_max_sharpe_capped(mu, cov, max_weight=C4_MAX_WEIGHT):
    n = len(mu)
    x0 = c4_equal_weights(n)

    bounds = [(0.0, max_weight) for _ in range(n)]
    constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1.0}]

    def objective(w):
        port_mu = float(w @ mu)
        port_vol = float(np.sqrt(max(w.T @ cov @ w, 1e-12)))
        return -((port_mu - C4_RISK_FREE_DAILY) / port_vol)

    res = minimize(
        objective,
        x0,
        method="SLSQP",
        bounds=bounds,
        constraints=constraints,
        options={"maxiter": 500}
    )

    if not res.success or np.any(~np.isfinite(res.x)):
        return x0

    w = np.clip(res.x, 0.0, max_weight)

    if w.sum() <= 0:
        return x0

    return w / w.sum()


def c4_build_weights(strategy_name):
    """
    Construye pesos diarios.

    La diferencia importante respecto a la versión anterior es que:
    - usamos datos previos a 2021 para estimar;
    - calculamos una cartera inicial justo antes de 2021;
    - evaluamos resultados desde 2021.
    """
    dates = prices_clean.index
    columns = prices_clean.columns.tolist()

    weights = pd.DataFrame(np.nan, index=dates, columns=columns)

    analysis_start_ts = pd.Timestamp(C4_ANALYSIS_START)

    analysis_dates = dates[dates >= analysis_start_ts]

    if len(analysis_dates) == 0:
        raise ValueError("No hay fechas disponibles dentro del periodo de análisis C4.")

    first_analysis_date = analysis_dates[0]

    # Fecha previa al análisis para dejar pesos calculados antes del primer día evaluado.
    pre_analysis_dates = dates[dates < first_analysis_date]

    if len(pre_analysis_dates) == 0:
        initial_rebalance_date = first_analysis_date
    else:
        initial_rebalance_date = pre_analysis_dates[-1]

    # Rebalanceos mensuales dentro del periodo de análisis.
    monthly_rebalance_dates = c4_get_rebalance_dates(
        dates,
        frequency=C4_REBALANCE_FREQUENCY,
        min_start_idx=C4_LOOKBACK_DAYS,
    )

    monthly_rebalance_dates = [
        d for d in monthly_rebalance_dates
        if d >= first_analysis_date and d <= pd.Timestamp(C4_ANALYSIS_END)
    ]

    rebalance_dates = sorted(set([initial_rebalance_date] + monthly_rebalance_dates))

    prev_weights = pd.Series(0.0, index=columns)
    n_failures = 0
    n_rebalances = 0

    for date in rebalance_dates:

        # Si el rebalanceo inicial es previo al análisis, usamos la membership
        # vigente al inicio del análisis, no una membership de 2020.
        if date < first_analysis_date:
            membership_date = first_analysis_date
        else:
            membership_date = date

        active_assets = c4_active_assets_on_date(membership_date, columns)

        if len(active_assets) < CONFIG.min_assets_required:
            weights.loc[date] = prev_weights
            n_failures += 1
            continue

        est_assets, mu, cov = c4_prepare_estimation_window(date, active_assets)

        if est_assets is None:
            weights.loc[date] = prev_weights
            n_failures += 1
            continue

        n = len(est_assets)

        if strategy_name == "EqualWeight":
            w_local = c4_equal_weights(n)

        elif strategy_name == "Markowitz_MaxSharpe_Capped_CML":
            base_weights = c4_optimize_max_sharpe_capped(
                mu,
                cov,
                max_weight=C4_MAX_WEIGHT
            )

            # Capital Market Line: en la ejecución final L=1.
            w_local = base_weights * C4_CML_LEVERAGE

        else:
            raise ValueError(f"Estrategia no reconocida: {strategy_name}")

        target = pd.Series(0.0, index=columns)
        target.loc[est_assets] = w_local

        weights.loc[date] = target
        prev_weights = target.copy()
        n_rebalances += 1

    weights = weights.ffill().fillna(0.0)

    analysis_weights = weights.loc[C4_ANALYSIS_START:C4_ANALYSIS_END]

    meta = {
        "strategy": strategy_name,
        "n_rebalances": int(n_rebalances),
        "n_failures": int(n_failures),
        "avg_n_positions": float((analysis_weights.abs() > 0).sum(axis=1).mean()),
        "avg_abs_exposure": float(analysis_weights.abs().sum(axis=1).mean()),
    }

    return weights, meta


def c4_backtest_weights(weights):
    """
    Backtest con cash explícito.

    Si exposición = 1:
        cartera totalmente invertida.

    Si exposición > 1:
        cartera apalancada, financiada al tipo libre de riesgo.

    Si exposición < 1:
        parte no invertida en cash.
    """
    held_weights = weights.shift(1).fillna(0.0)

    risky_returns = (held_weights * asset_returns).sum(axis=1)

    gross_exposure = held_weights.sum(axis=1)
    cash_weight = 1.0 - gross_exposure

    borrow_spread_daily = (
        (1.0 + C4_BORROW_SPREAD_ANNUAL) ** (1.0 / CONFIG.trading_days_per_year)
        - 1.0
    )

    financing_cost = np.maximum(-cash_weight, 0.0) * borrow_spread_daily

    gross_returns = (
        risky_returns
        + cash_weight * C4_RISK_FREE_DAILY
        - financing_cost
    )

    turnover = weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = weights.iloc[0].abs().sum()
    turnover = turnover.fillna(0.0)

    costs = C4_TRANSACTION_COST * turnover

    net_returns = gross_returns - costs
    net_returns.name = "strategy_returns"

    equity = c4_equity_from_returns(net_returns, CONFIG.initial_capital)
    equity.name = "strategy_equity"

    return net_returns, equity, turnover


In [ ]:
# ============================================================
# Celda C4.5 - Ejecutar backtest definitivo Markowitz-CML


In [ ]:
# ============================================================

C4_STRATEGIES = [
    "EqualWeight",
    "Markowitz_MaxSharpe_Capped_CML",
]

strategy_returns_dict = {}
strategy_equity_dict = {}
strategy_weights_dict = {}
strategy_turnover_dict = {}
strategy_meta_rows = []

for strategy_name in C4_STRATEGIES:
    print("Ejecutando:", strategy_name)

    weights, meta = c4_build_weights(strategy_name)
    returns, equity, turnover = c4_backtest_weights(weights)

    strategy_returns_dict[strategy_name] = returns
    strategy_equity_dict[strategy_name] = equity
    strategy_weights_dict[strategy_name] = weights
    strategy_turnover_dict[strategy_name] = turnover

    analysis_turnover = turnover.loc[C4_ANALYSIS_START:C4_ANALYSIS_END]

    meta["total_turnover"] = float(analysis_turnover.sum())
    meta["avg_turnover"] = float(analysis_turnover.mean())
    strategy_meta_rows.append(meta)

strategy_returns_full = pd.DataFrame(strategy_returns_dict).sort_index()
strategy_meta = pd.DataFrame(strategy_meta_rows)

# ------------------------------------------------------------
# Forzar periodo común de análisis 2021-2025
# ------------------------------------------------------------

analysis_index = strategy_returns_full.index[
    (strategy_returns_full.index >= pd.Timestamp(C4_ANALYSIS_START)) &
    (strategy_returns_full.index <= pd.Timestamp(C4_ANALYSIS_END))
]

analysis_index = analysis_index.intersection(benchmark_returns.index)

if len(analysis_index) == 0:
    raise ValueError("No hay fechas comunes para el periodo de análisis C4.")

C4_VALID_START = analysis_index.min()
C4_VALID_END = analysis_index.max()

strategy_returns = strategy_returns_full.loc[analysis_index].copy()
strategy_equity = (1.0 + strategy_returns).cumprod()

benchmark_returns_c4 = benchmark_returns.loc[analysis_index].fillna(0.0)
benchmark_returns_c4.name = "Benchmark_SPY"
benchmark_equity_c4 = c4_equity_from_returns(benchmark_returns_c4)

# ------------------------------------------------------------
# Métricas en periodo común
# ------------------------------------------------------------

metrics_rows = []

for col in strategy_returns.columns:
    metrics_rows.append(c4_compute_metrics(strategy_returns[col], col))

metrics_rows.append(c4_compute_metrics(benchmark_returns_c4, "Benchmark_SPY"))

metrics = pd.DataFrame(metrics_rows)

selected_strategy = "Markowitz_MaxSharpe_Capped_CML"

selected_returns = strategy_returns[selected_strategy].copy()
selected_returns.name = selected_strategy
selected_equity = strategy_equity[selected_strategy].copy()

equalweight_returns = strategy_returns["EqualWeight"].copy()
equalweight_returns.name = "EqualWeight"
equalweight_equity = strategy_equity["EqualWeight"].copy()

print("Periodo de análisis C4:", C4_VALID_START.date(), "->", C4_VALID_END.date())
print("Estrategia principal:", selected_strategy)

display(metrics)
display(strategy_meta)

# ------------------------------------------------------------
# Guardar resultados
# ------------------------------------------------------------

strategy_returns.to_csv(C4_TABLES_DIR / "C4_strategy_returns.csv")
strategy_equity.to_csv(C4_TABLES_DIR / "C4_strategy_equity.csv")
metrics.to_csv(C4_TABLES_DIR / "C4_metrics.csv", index=False)
strategy_meta.to_csv(C4_TABLES_DIR / "C4_strategy_metadata.csv", index=False)

for name, w in strategy_weights_dict.items():
    w.loc[analysis_index].to_csv(C4_TABLES_DIR / f"C4_weights_{name}.csv")
    strategy_turnover_dict[name].loc[analysis_index].to_csv(
        C4_TABLES_DIR / f"C4_turnover_{name}.csv"
    )


In [ ]:
# ============================================================
# Celda C4.6 - Bootstrap


In [ ]:
# ============================================================

def c4_moving_block_bootstrap_indices(n, block_length, rng):
    starts = rng.integers(
        0,
        max(1, n - block_length + 1),
        size=int(np.ceil(n / block_length))
    )

    idx = []

    for s in starts:
        idx.extend(range(s, min(s + block_length, n)))

    return np.array(idx[:n])


def c4_block_bootstrap_metrics(
    returns,
    n_bootstrap=C4_N_BOOTSTRAP,
    block_length=C4_BLOCK_LENGTH,
    seed=C4_RANDOM_SEED
):
    rng_local = np.random.default_rng(seed)

    r = pd.Series(returns).dropna().values
    rows = []

    for b in range(n_bootstrap):
        idx = c4_moving_block_bootstrap_indices(len(r), block_length, rng_local)
        sample = pd.Series(r[idx])
        equity = c4_equity_from_returns(sample)

        rows.append(
            compute_metrics(sample, equity, CONFIG, f"bootstrap_{b}")
        )

    return pd.DataFrame(rows)


def c4_bootstrap_percentiles(boot_df, label):
    qs = boot_df[[
        "CAGR",
        "Sharpe",
        "Sortino",
        "max_drawdown",
        "Calmar"
    ]].quantile([0.025, 0.05, 0.50, 0.95, 0.975])

    qs.index = ["p2_5", "p5", "p50", "p95", "p97_5"]
    qs.insert(0, "strategy", label)

    return qs.reset_index(names="percentile")


boot_selected = c4_block_bootstrap_metrics(
    selected_returns,
    seed=C4_RANDOM_SEED + 1
)

boot_equalweight = c4_block_bootstrap_metrics(
    equalweight_returns,
    seed=C4_RANDOM_SEED + 2
)

boot_benchmark = c4_block_bootstrap_metrics(
    benchmark_returns_c4,
    seed=C4_RANDOM_SEED + 3
)

bootstrap_percentiles = pd.concat([
    c4_bootstrap_percentiles(boot_selected, selected_strategy),
    c4_bootstrap_percentiles(boot_equalweight, "EqualWeight"),
    c4_bootstrap_percentiles(boot_benchmark, "Benchmark_SPY"),
], ignore_index=True)

print("=== Bootstrap percentiles ===")
display(bootstrap_percentiles)

boot_selected.to_csv(C4_TABLES_DIR / "C4_bootstrap_selected.csv", index=False)
boot_equalweight.to_csv(C4_TABLES_DIR / "C4_bootstrap_equalweight.csv", index=False)
boot_benchmark.to_csv(C4_TABLES_DIR / "C4_bootstrap_benchmark.csv", index=False)
bootstrap_percentiles.to_csv(C4_TABLES_DIR / "C4_bootstrap_percentiles.csv", index=False)


In [ ]:
# ============================================================
# Celda C4.6c - Bootstrap temporal all-in-one CORREGIDO
# Usa directamente las series correctas del notebook


In [ ]:
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

C4_BOOTSTRAP_N_TEMPORAL = 1000
C4_BOOTSTRAP_BLOCK_LENGTH_TEMPORAL = 20
C4_BOOTSTRAP_SEED_TEMPORAL = 123

# ------------------------------------------------------------
# 1. Cargar series correctas desde memoria
# ------------------------------------------------------------
# Estas variables deberían existir en tu C4:
# selected_returns        -> Markowitz
# equalweight_returns     -> EqualWeight
# benchmark_returns_c4    -> Benchmark SPY

required_vars = [
    "selected_returns",
    "equalweight_returns",
    "benchmark_returns_c4",
]

missing = [v for v in required_vars if v not in globals()]

if missing:
    raise NameError(
        f"Faltan variables en memoria: {missing}. "
        "Ejecuta primero las celdas anteriores de C4 hasta la parte de resultados principales."
    )

returns_df = pd.concat(
    [
        pd.Series(selected_returns, name="Markowitz"),
        pd.Series(equalweight_returns, name="EqualWeight"),
        pd.Series(benchmark_returns_c4, name="Benchmark"),
    ],
    axis=1
).dropna()

returns_df = returns_df.replace([np.inf, -np.inf], np.nan).dropna()

print("Series usadas para el bootstrap temporal:")
display(returns_df.head())
display(returns_df.tail())

# ------------------------------------------------------------
# 2. Check de coherencia con tus números principales
# ------------------------------------------------------------

def c4_basic_check_metrics(r, trading_days=252):
    r = pd.Series(r).dropna().astype(float)
    cumulative_return = (1.0 + r).prod() - 1.0
    years = len(r) / trading_days
    cagr = (1.0 + cumulative_return) ** (1.0 / years) - 1.0
    return cumulative_return, cagr

check_rows = []

for col in returns_df.columns:
    cum_ret, cagr = c4_basic_check_metrics(returns_df[col])
    check_rows.append({
        "Estrategia": col,
        "Ret. acumulada": cum_ret,
        "CAGR": cagr,
    })

check_df = pd.DataFrame(check_rows)

print("Comprobación rápida. Esto debería parecerse a la tabla del Capítulo 5:")
display(check_df)

markowitz_cum = check_df.loc[
    check_df["Estrategia"] == "Markowitz",
    "Ret. acumulada"
].iloc[0]

if markowitz_cum > 1.50:
    print(
        "AVISO: Markowitz tiene más de 150% de rentabilidad acumulada. "
        "Probablemente estás usando la versión apalancada."
    )

if markowitz_cum < 0.90:
    print(
        "AVISO: Markowitz no parece estar usando la serie correcta. "
        "La rentabilidad acumulada esperada sin apalancamiento ronda el 116%."
    )

# ------------------------------------------------------------
# 3. Bootstrap por bloques preservando fechas comunes
# ------------------------------------------------------------

def c4_bootstrap_equity_paths(returns_df, n_bootstrap=1000, block_length=20, seed=123):
    rng = np.random.default_rng(seed)

    values = returns_df.values
    n, k = values.shape

    paths = {
        col: np.zeros((n_bootstrap, n))
        for col in returns_df.columns
    }

    for b in range(n_bootstrap):
        sample_blocks = []

        while len(sample_blocks) < n:
            start = rng.integers(0, n - block_length + 1)
            block = values[start:start + block_length, :]
            sample_blocks.extend(block)

        sample = np.array(sample_blocks[:n])
        equity = np.cumprod(1.0 + sample, axis=0)

        for j, col in enumerate(returns_df.columns):
            paths[col][b, :] = equity[:, j]

    return paths


bootstrap_paths = c4_bootstrap_equity_paths(
    returns_df,
    n_bootstrap=C4_BOOTSTRAP_N_TEMPORAL,
    block_length=C4_BOOTSTRAP_BLOCK_LENGTH_TEMPORAL,
    seed=C4_BOOTSTRAP_SEED_TEMPORAL
)

# ------------------------------------------------------------
# 4. Calcular percentiles temporales
# ------------------------------------------------------------

percentile_tables = {}

for strategy, arr in bootstrap_paths.items():
    p2 = np.percentile(arr, 2.5, axis=0)
    p50 = np.percentile(arr, 50.0, axis=0)
    p97 = np.percentile(arr, 97.5, axis=0)

    actual_equity = (1.0 + returns_df[strategy]).cumprod()

    out = pd.DataFrame({
        "date": returns_df.index,
        "actual_equity": actual_equity.values,
        "p2_5": p2,
        "p50": p50,
        "p97_5": p97,
    })

    percentile_tables[strategy] = out

    safe_name = strategy.replace("&", "and").replace(" ", "_").replace("/", "_")

    out.to_csv(
        C4_TABLES_DIR / f"C4_bootstrap_equity_percentiles_{safe_name}.csv",
        index=False
    )

# Tabla conjunta para revisar si hace falta
combined_rows = []

for strategy, out in percentile_tables.items():
    temp = out.copy()
    temp.insert(0, "strategy", strategy)
    combined_rows.append(temp)

combined_percentiles = pd.concat(combined_rows, ignore_index=True)

combined_percentiles.to_csv(
    C4_TABLES_DIR / "C4_bootstrap_equity_percentiles_all_in_one.csv",
    index=False
)

# ------------------------------------------------------------
# 5. Figura all-in-one: 3 estrategias x 3 percentiles
# ------------------------------------------------------------

plt.figure(figsize=(10.5, 6))

strategy_order = ["Markowitz", "EqualWeight", "Benchmark"]

default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]

strategy_colors = {
    strategy: default_colors[i % len(default_colors)]
    for i, strategy in enumerate(strategy_order)
}

linestyle_map = {
    "p2_5": "--",
    "p50": "-",
    "p97_5": ":",
}

label_map = {
    "p2_5": "Percentil 2,5",
    "p50": "Percentil 50",
    "p97_5": "Percentil 97,5",
}

for strategy in strategy_order:
    out = percentile_tables[strategy]
    color = strategy_colors[strategy]

    for key in ["p2_5", "p50", "p97_5"]:
        plt.plot(
            range(len(out)),
            out[key],
            linestyle=linestyle_map[key],
            linewidth=2.4 if key == "p50" else 1.5,
            alpha=1.0 if key == "p50" else 0.80,
            color=color
        )

plt.title("Evolución temporal bootstrap de Markowitz")
plt.xlabel("Tiempo en el periodo operativo")
plt.ylabel("Capital normalizado")
plt.grid(axis="y", alpha=0.3)

strategy_handles = [
    Line2D(
        [0], [0],
        color=strategy_colors[s],
        lw=2.6,
        label=s
    )
    for s in strategy_order
]

percentile_handles = [
    Line2D(
        [0], [0],
        color="black",
        lw=2.4 if k == "p50" else 1.5,
        linestyle=linestyle_map[k],
        label=label_map[k]
    )
    for k in ["p2_5", "p50", "p97_5"]
]

legend1 = plt.legend(
    handles=strategy_handles,
    title="Estrategia",
    loc="upper left"
)
plt.gca().add_artist(legend1)

plt.legend(
    handles=percentile_handles,
    title="Curva",
    loc="upper right"
)

plt.tight_layout()

fig_path = C4_FIGURES_DIR / "C4_bootstrap_equity_percentiles_all_in_one.png"
plt.savefig(fig_path, dpi=160)
plt.show()

print("Archivos guardados:")
print(C4_TABLES_DIR / "C4_bootstrap_equity_percentiles_all_in_one.csv")
print(C4_FIGURES_DIR / "C4_bootstrap_equity_percentiles_all_in_one.png")


In [ ]:
# ============================================================
# Celda C4.7 - Regresión factorial FF5 + MOM


In [ ]:
# ============================================================

def c4_run_factor_regression(returns_dict, factors_df):
    rows = []
    coef_rows = []

    factor_cols = ["Mkt_RF", "SMB", "HML", "RMW", "CMA", "MOM"]

    for name, r in returns_dict.items():
        r = pd.Series(r).dropna()

        common = r.index.intersection(factors_df.index)

        y_raw = r.loc[common] * 100.0
        f = factors_df.loc[common].copy()

        y = y_raw - f["RF"]

        x = f[factor_cols]
        x = sm.add_constant(x)

        data = pd.concat(
            [y.rename("excess_return"), x],
            axis=1
        ).dropna()

        if len(data) < 60:
            continue

        model = sm.OLS(
            data["excess_return"],
            data.drop(columns=["excess_return"])
        ).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": 5}
        )

        alpha_daily_pct = float(model.params["const"])
        alpha_annual_pct = alpha_daily_pct * CONFIG.trading_days_per_year

        rows.append({
            "strategy": name,
            "n_obs": int(len(data)),
            "alpha_daily_pct": alpha_daily_pct,
            "alpha_annual_pct": alpha_annual_pct,
            "alpha_tstat": float(model.tvalues["const"]),
            "alpha_pvalue": float(model.pvalues["const"]),
            "r_squared": float(model.rsquared),
            "adj_r_squared": float(model.rsquared_adj),
        })

        for param in model.params.index:
            coef_rows.append({
                "strategy": name,
                "factor": param,
                "coef": float(model.params[param]),
                "tstat": float(model.tvalues[param]),
                "pvalue": float(model.pvalues[param]),
            })

    return pd.DataFrame(rows), pd.DataFrame(coef_rows)


regression_returns = {
    selected_strategy: selected_returns,
    "EqualWeight": equalweight_returns,
    "Benchmark_SPY": benchmark_returns_c4,
}

factor_summary, factor_coefficients = c4_run_factor_regression(
    regression_returns,
    factors
)

print("=== Resumen regresión FF5+MOM ===")
display(factor_summary)

print("=== Coeficientes FF5+MOM ===")
display(factor_coefficients)

factor_summary.to_csv(
    C4_TABLES_DIR / "C4_factor_regression_summary.csv",
    index=False
)

factor_coefficients.to_csv(
    C4_TABLES_DIR / "C4_factor_regression_coefficients.csv",
    index=False
)


In [ ]:
# ============================================================
# Celda C4.8 - Figuras, tablas LaTeX y metadata


In [ ]:
# ============================================================

def c4_fmt_pct(x):
    if pd.isna(x):
        return ""
    return f"{100.0 * float(x):.2f}\\%".replace(".", ",")


def c4_fmt_pct_already(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.2f}\\%".replace(".", ",")


def c4_fmt_num(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.3f}".replace(".", ",")


def c4_fmt_p(x):
    if pd.isna(x):
        return ""
    return f"{float(x):.3f}".replace(".", ",")


# ------------------------------------------------------------
# Figuras
# ------------------------------------------------------------

plot_equity = pd.DataFrame({
    selected_strategy: c4_equity_from_returns(selected_returns),
    "EqualWeight": c4_equity_from_returns(equalweight_returns),
    "Benchmark_SPY": c4_equity_from_returns(benchmark_returns_c4),
}).dropna()

plot_equity = plot_equity / plot_equity.iloc[0]

plt.figure(figsize=(12, 6))
plot_equity.plot(ax=plt.gca())
plt.title("Módulo C4: Markowitz frente a referencias")
plt.xlabel("Fecha")
plt.ylabel("Capital normalizado")
plt.legend()
plt.tight_layout()
plt.savefig(C4_FIGURES_DIR / "C4_equity_curves_main.png", dpi=160)
plt.close()

plt.figure(figsize=(12, 6))
for col in plot_equity.columns:
    c4_drawdown_series(plot_equity[col]).plot(label=col)
plt.title("Módulo C4: drawdowns")
plt.xlabel("Fecha")
plt.ylabel("Drawdown")
plt.legend()
plt.tight_layout()
plt.savefig(C4_FIGURES_DIR / "C4_drawdowns_main.png", dpi=160)
plt.close()

factor_bar = factor_summary.copy().sort_values("alpha_annual_pct", ascending=True)

plt.figure(figsize=(10, 6))
plt.barh(factor_bar["strategy"], factor_bar["alpha_annual_pct"])
plt.axvline(0.0, linewidth=1)
plt.title("Módulo C4: alpha anualizado FF5+MOM")
plt.xlabel("Alpha anualizado (%)")
plt.tight_layout()
plt.savefig(C4_FIGURES_DIR / "C4_factor_alpha_bar.png", dpi=160)
plt.close()

exposure = pd.DataFrame({
    name: w.loc[strategy_returns.index].abs().sum(axis=1)
    for name, w in strategy_weights_dict.items()
})

plt.figure(figsize=(12, 5))
exposure.plot(ax=plt.gca())
plt.title("Módulo C4: exposición bruta")
plt.xlabel("Fecha")
plt.ylabel("Exposición")
plt.legend()
plt.tight_layout()
plt.savefig(C4_FIGURES_DIR / "C4_gross_exposure.png", dpi=160)
plt.close()

# ------------------------------------------------------------
# Tablas LaTeX
# ------------------------------------------------------------

latex_main = metrics[[
    "label",
    "cumulative_return",
    "CAGR",
    "Sharpe",
    "Sortino",
    "max_drawdown",
    "Calmar",
]].copy()

latex_main["cumulative_return"] = latex_main["cumulative_return"].map(c4_fmt_pct)
latex_main["CAGR"] = latex_main["CAGR"].map(c4_fmt_pct)
latex_main["Sharpe"] = latex_main["Sharpe"].map(c4_fmt_num)
latex_main["Sortino"] = latex_main["Sortino"].map(c4_fmt_num)
latex_main["max_drawdown"] = latex_main["max_drawdown"].map(c4_fmt_pct)
latex_main["Calmar"] = latex_main["Calmar"].map(c4_fmt_num)

latex_main = latex_main.rename(columns={
    "label": "Estrategia",
    "cumulative_return": "Ret. acumulada",
    "max_drawdown": "MDD",
})

latex_main.to_latex(
    C4_TABLES_DIR / "C4_table_main_metrics.tex",
    index=False,
    escape=False,
    caption="Comparación de Markowitz frente a EqualWeight y benchmark.",
    label="tab:c4_main_metrics"
)

latex_factor = factor_summary.copy()

latex_factor["alpha_annual_pct"] = latex_factor["alpha_annual_pct"].map(c4_fmt_pct_already)
latex_factor["alpha_tstat"] = latex_factor["alpha_tstat"].map(c4_fmt_num)
latex_factor["alpha_pvalue"] = latex_factor["alpha_pvalue"].map(c4_fmt_p)
latex_factor["r_squared"] = latex_factor["r_squared"].map(c4_fmt_num)

latex_factor = latex_factor[[
    "strategy",
    "alpha_annual_pct",
    "alpha_tstat",
    "alpha_pvalue",
    "r_squared",
]].rename(columns={
    "strategy": "Estrategia",
    "alpha_annual_pct": "Alpha anualizado",
    "alpha_tstat": "t-stat alpha",
    "alpha_pvalue": "p-value alpha",
    "r_squared": "$R^2$",
})

latex_factor.to_latex(
    C4_TABLES_DIR / "C4_table_factor_summary.tex",
    index=False,
    escape=False,
    caption="Regresión factorial FF5+MOM para Markowitz y referencias.",
    label="tab:c4_factor_summary"
)

# ------------------------------------------------------------
# Metadata
# ------------------------------------------------------------

metadata = {
    "module": "C4_markowitz_cml_final_2021_2025",
    "description": "Evaluación de una estrategia Markowitz predefinida: cartera MaxSharpe con peso máximo. La evaluación se realiza en 2021-2025 usando datos previos solo como ventana inicial de estimación.",
    "data_start": C4_DATA_START,
    "analysis_start": C4_ANALYSIS_START,
    "analysis_end": C4_ANALYSIS_END,
    "valid_start": str(C4_VALID_START.date()),
    "valid_end": str(C4_VALID_END.date()),
    "universe": "S&P 100 historical reconstruction from C0",
    "benchmark_ticker": CONFIG.benchmark_ticker,
    "transaction_cost": C4_TRANSACTION_COST,
    "risk_free_rate_annual": C4_RISK_FREE_RATE_ANNUAL,
    "lookback_days": C4_LOOKBACK_DAYS,
    "rebalance_frequency": C4_REBALANCE_FREQUENCY,
    "max_weight": C4_MAX_WEIGHT,
    "cml_leverage": C4_CML_LEVERAGE,
    "borrow_spread_annual": C4_BORROW_SPREAD_ANNUAL,
    "selected_strategy": selected_strategy,
    "n_assets": int(prices_clean.shape[1]),
    "n_observations": int(strategy_returns.shape[0]),
    "bootstrap_n": C4_N_BOOTSTRAP,
    "block_length": C4_BLOCK_LENGTH,
    "white_reality_check": "not_applied_single_prespecified_strategy",
    "factor_source_ff5": FF5_DAILY_URL,
    "factor_source_mom": MOM_DAILY_URL,
}

with open(C4_OUTPUT_DIR / "C4_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4, ensure_ascii=False)

print("Resultados guardados en:", C4_OUTPUT_DIR)
print("Tablas:", C4_TABLES_DIR)
print("Figuras:", C4_FIGURES_DIR)


In [ ]:
# ============================================================
# Celda C4.8b - Sensibilidad al coste de financiación del apalancamiento


In [ ]:
# ============================================================

C4_BORROW_SPREAD_GRID = [0.00, 0.01, 0.02, 0.04, 0.06]

def c4_backtest_weights_with_borrow_spread(weights, borrow_spread_annual):
    held_weights = weights.shift(1).fillna(0.0)

    risky_returns = (held_weights * asset_returns).sum(axis=1)

    gross_exposure = held_weights.sum(axis=1)
    cash_weight = 1.0 - gross_exposure

    borrow_spread_daily = (
        (1.0 + borrow_spread_annual) ** (1.0 / CONFIG.trading_days_per_year)
        - 1.0
    )

    financing_cost = np.maximum(-cash_weight, 0.0) * borrow_spread_daily

    gross_returns = (
        risky_returns
        + cash_weight * C4_RISK_FREE_DAILY
        - financing_cost
    )

    turnover = weights.diff().abs().sum(axis=1)
    turnover.iloc[0] = weights.iloc[0].abs().sum()
    turnover = turnover.fillna(0.0)

    costs = C4_TRANSACTION_COST * turnover

    net_returns = gross_returns - costs
    net_returns.name = "strategy_returns"

    equity = c4_equity_from_returns(net_returns, CONFIG.initial_capital)
    equity.name = "strategy_equity"

    return net_returns, equity, turnover


sensitivity_rows = []
sensitivity_returns = {}

base_weights = strategy_weights_dict[selected_strategy].copy()

for spread in C4_BORROW_SPREAD_GRID:
    r_spread, eq_spread, turnover_spread = c4_backtest_weights_with_borrow_spread(
        base_weights,
        borrow_spread_annual=spread
    )

    r_spread = r_spread.loc[r_spread.index >= C4_VALID_START].copy()
    eq_spread = c4_equity_from_returns(r_spread, CONFIG.initial_capital)

    label = f"spread_{int(spread * 100)}pct"
    sensitivity_returns[label] = r_spread

    m = compute_metrics(
        r_spread,
        eq_spread,
        CONFIG,
        label
    )

    m["borrow_spread_annual"] = spread
    m["financing_rate_annual"] = C4_RISK_FREE_RATE_ANNUAL + spread
    sensitivity_rows.append(m)

borrow_sensitivity = pd.DataFrame(sensitivity_rows)

borrow_sensitivity = borrow_sensitivity[[
    "borrow_spread_annual",
    "financing_rate_annual",
    "cumulative_return",
    "CAGR",
    "Sharpe",
    "Sortino",
    "max_drawdown",
    "Calmar",
]]

print("Sensibilidad al coste de financiación:")
display(borrow_sensitivity)

borrow_sensitivity.to_csv(
    C4_TABLES_DIR / "C4_borrow_spread_sensitivity.csv",
    index=False
)

# Tabla LaTeX
latex_borrow = borrow_sensitivity.copy()

latex_borrow["borrow_spread_annual"] = latex_borrow["borrow_spread_annual"].map(c4_fmt_pct)
latex_borrow["financing_rate_annual"] = latex_borrow["financing_rate_annual"].map(c4_fmt_pct)
latex_borrow["cumulative_return"] = latex_borrow["cumulative_return"].map(c4_fmt_pct)
latex_borrow["CAGR"] = latex_borrow["CAGR"].map(c4_fmt_pct)
latex_borrow["Sharpe"] = latex_borrow["Sharpe"].map(c4_fmt_num)
latex_borrow["Sortino"] = latex_borrow["Sortino"].map(c4_fmt_num)
latex_borrow["max_drawdown"] = latex_borrow["max_drawdown"].map(c4_fmt_pct)
latex_borrow["Calmar"] = latex_borrow["Calmar"].map(c4_fmt_num)

latex_borrow = latex_borrow.rename(columns={
    "borrow_spread_annual": "Spread financiación",
    "financing_rate_annual": "Tipo financiación",
    "cumulative_return": "Ret. acumulada",
    "max_drawdown": "MDD",
})

latex_borrow.to_latex(
    C4_TABLES_DIR / "C4_table_borrow_spread_sensitivity.tex",
    index=False,
    escape=False,
    caption="Sensibilidad de Markowitz-CML al coste de financiación del apalancamiento.",
    label="tab:c4_borrow_spread_sensitivity"
)

# Figura CAGR y Calmar
plt.figure(figsize=(9, 5))
plt.plot(
    borrow_sensitivity["borrow_spread_annual"] * 100,
    borrow_sensitivity["CAGR"] * 100,
    marker="o"
)
plt.title("Sensibilidad de la CAGR al coste de financiación")
plt.xlabel("Spread adicional de financiación (%)")
plt.ylabel("CAGR (%)")
plt.tight_layout()
plt.savefig(C4_FIGURES_DIR / "C4_borrow_spread_sensitivity_CAGR.png", dpi=160)
plt.close()

plt.figure(figsize=(9, 5))
plt.plot(
    borrow_sensitivity["borrow_spread_annual"] * 100,
    borrow_sensitivity["Calmar"],
    marker="o"
)
plt.title("Sensibilidad del Calmar al coste de financiación")
plt.xlabel("Spread adicional de financiación (%)")
plt.ylabel("Calmar")
plt.tight_layout()
plt.savefig(C4_FIGURES_DIR / "C4_borrow_spread_sensitivity_Calmar.png", dpi=160)
plt.close()

print("Guardado:")
print(C4_TABLES_DIR / "C4_borrow_spread_sensitivity.csv")
print(C4_TABLES_DIR / "C4_table_borrow_spread_sensitivity.tex")
print(C4_FIGURES_DIR / "C4_borrow_spread_sensitivity_CAGR.png")
print(C4_FIGURES_DIR / "C4_borrow_spread_sensitivity_Calmar.png")


In [ ]:
# ============================================================
# Celda C4.9 - Ver resultados y descargar ZIP


In [ ]:
# ============================================================

from IPython.display import Image, display
import shutil

print("============================================================")
print("MÓDULO C4 - RESUMEN FINAL")
print("============================================================")
print("Estrategia principal:", selected_strategy)
print("Periodo de análisis:", C4_VALID_START.date(), "->", C4_VALID_END.date())
print("Datos usados desde:", C4_DATA_START)
print("Carpeta:", C4_OUTPUT_DIR)

print("\n=== Métricas ===")
display(metrics)

print("\n=== Bootstrap percentiles ===")
display(bootstrap_percentiles)

print("\n=== Regresión FF5+MOM ===")
display(factor_summary)

print("\nNota metodológica:")
print("No se aplica White's Reality Check porque se evalúa una única estrategia Markowitz preespecificada.")
print("White se reserva para casos con búsqueda múltiple de estrategias.")
print("Los datos previos a 2021 se usan solo para estimar la primera cartera, no como periodo de evaluación.")

figures_to_show = [
    "C4_equity_curves_main.png",
    "C4_drawdowns_main.png",
    "C4_factor_alpha_bar.png",
    "C4_gross_exposure.png",
    "C4_bootstrap_equity_percentiles_all_in_one.png",
]

for fig in figures_to_show:
    path = C4_FIGURES_DIR / fig

    if path.exists():
        print("\n", fig)
        display(Image(filename=str(path)))
    else:
        print("No encontrada:", fig)

zip_base_name = "C4_markowitz_cml_final_2021_2025_results"
zip_path = Path(str(C4_OUTPUT_DIR.parent / zip_base_name) + ".zip")

if zip_path.exists():
    zip_path.unlink()

shutil.make_archive(
    base_name=str(C4_OUTPUT_DIR.parent / zip_base_name),
    format="zip",
    root_dir=str(C4_OUTPUT_DIR),
)

print("\nZIP creado en:")
print(zip_path)

try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print("Descarga automática no disponible. Descarga manualmente desde:")
    print(zip_path)
    print("Error:", e)


In [ ]:
# ============================================================
# Celda C4.10 - Resumen rápido C4


In [ ]:
# ============================================================

def pct(x):
    return f"{100 * float(x):.2f}%"

def num(x):
    return f"{float(x):.3f}"

print("============================================================")
print("C4 - RESUMEN RÁPIDO")
print("============================================================")

print("\nEstrategia principal:")
print(selected_strategy)

print("\nPeriodo de análisis:")
print(C4_VALID_START.date(), "->", C4_VALID_END.date())

print("\nDatos previos usados para estimación inicial:")
print(C4_DATA_START, "->", C4_ANALYSIS_START)

print("\n1) MÉTRICAS")
main = metrics[[
    "label",
    "cumulative_return",
    "CAGR",
    "Sharpe",
    "Sortino",
    "max_drawdown",
    "Calmar"
]].copy()

main["cumulative_return"] = main["cumulative_return"].map(pct)
main["CAGR"] = main["CAGR"].map(pct)
main["Sharpe"] = main["Sharpe"].map(num)
main["Sortino"] = main["Sortino"].map(num)
main["max_drawdown"] = main["max_drawdown"].map(pct)
main["Calmar"] = main["Calmar"].map(num)

display(main)

print("\n2) FF5 + MOM")
factor_short = factor_summary[[
    "strategy",
    "alpha_annual_pct",
    "alpha_tstat",
    "alpha_pvalue",
    "r_squared"
]].copy()

factor_short["alpha_annual_pct"] = factor_short["alpha_annual_pct"].map(lambda x: f"{float(x):.2f}%")
factor_short["alpha_tstat"] = factor_short["alpha_tstat"].map(num)
factor_short["alpha_pvalue"] = factor_short["alpha_pvalue"].map(num)
factor_short["r_squared"] = factor_short["r_squared"].map(num)

display(factor_short)

selected_alpha = factor_summary[factor_summary["strategy"] == selected_strategy]

if not selected_alpha.empty:
    p_alpha = float(selected_alpha.iloc[0]["alpha_pvalue"])
    alpha = float(selected_alpha.iloc[0]["alpha_annual_pct"])

    if p_alpha < 0.05:
        print(f"Alpha significativo para {selected_strategy}: {alpha:.2f}% anual | p = {p_alpha:.3f}")
    else:
        print(f"Alpha NO significativo para {selected_strategy}: {alpha:.2f}% anual | p = {p_alpha:.3f}")

print("\n3) WHITE")
print("No se aplica White's Reality Check en C4.")
print("Motivo: se evalúa una única estrategia Markowitz predefinida, no una familia de estrategias seleccionadas ex post.")

print("\n4) FIGURAS PRINCIPALES")
print("- C4_equity_curves_main.png")
print("- C4_drawdowns_main.png")
print("- C4_bootstrap_equity_percentiles_all_in_one.png")
print("- C4_factor_alpha_bar.png")
print("- C4_gross_exposure.png")
